# USD/VND volatility forecasting pipeline — audited revision

This revision adds strict split-level data audits, chronology checks, auditable ARMA candidate failures, strict positive-variance validation, and standardized-residual diagnostics after volatility fitting on both the training and development samples.


In [1]:
# ============================================================
# 0. IMPORTS AND GLOBAL SETTINGS
# ============================================================

import sys
import subprocess
import importlib.util
import warnings
from pathlib import Path
from typing import Any
import statsmodels.api as sm
from statsmodels.tsa.api import VAR

warnings.filterwarnings("ignore")

if importlib.util.find_spec("arch") is None:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "arch",
        ]
    )

import numpy as np
import pandas as pd

from scipy.optimize import minimize
from scipy.special import gammaln

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import (
    acorr_ljungbox,
    het_arch,
)

from arch import arch_model

try:
    from IPython.display import display
except Exception:
    display = print

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


pd.set_option(
    "display.max_columns",
    150,
)

pd.set_option(
    "display.width",
    220,
)

pd.set_option(
    "display.float_format",
    lambda x: f"{x:,.6f}",
)


RANDOM_SEED = 42

np.random.seed(
    RANDOM_SEED
)

DATA_DIR = Path(
    "../data/processed"
)

OUTPUT_DIR = Path(
    "../outputs"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


TRAIN_PATH = (
    DATA_DIR
    / "model_data_train.csv"
)

VALID_PATH = (
    DATA_DIR
    / "model_data_valid.csv"
)

TEST_PATH = (
    DATA_DIR
    / "model_data_test.csv"
)

SHOCK_COLS = [
    "vix_change_lag1",
    "us10y_change_lag1",
    "dxy_return_lag1",
    "oil_change_lag1",
    "vnindex_return_lag1",
]

REQUIRED_COLS = [
    "Date",
    "fx_return",
    *SHOCK_COLS,
]

MAX_ARMA_P = 5
MAX_ARMA_Q = 5

ARMA_LB_LAGS = [
    10,
    20,
    30,
]

ARMA_MIN_LB_PVALUE = 0.05

MIN_FORECAST_VAR = 1e-12

EWMA_LAMBDA = 0.94


PARAMETRIC_MODELS = [
    "GARCH(1,1)-t",
    "GJR-GARCH(1,1)-t",
    "EGARCH(1,1)-t",
    "Custom EGARCH no-X-t",
    "EGARCH-X signed lag1-t",
    "EGARCH-X abs lag1-t",
]

BENCHMARK_MODELS = [
    "EWMA_0.94",
    "HistoricalVariance",
    "RollingVariance22",
]

ALL_VALIDATION_MODELS = [
    *PARAMETRIC_MODELS,
    *BENCHMARK_MODELS,
]


In [2]:

# ============================================================
# 1. LOAD DATA AND RUN STRICT DATA AUDIT
# ============================================================

NUMERIC_REQUIRED_COLS = [
    "fx_return",
    *SHOCK_COLS,
]


def load_split(
    path: Path,
    split_name: str,
) -> pd.DataFrame:
    """
    Load one chronological split and fail immediately when the
    required research inputs are malformed.
    """
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {split_name} dataset: {path.resolve()}"
        )

    df = pd.read_csv(path)

    missing_cols = sorted(
        set(REQUIRED_COLS) - set(df.columns)
    )

    if missing_cols:
        raise ValueError(
            f"{split_name}: missing required columns: {missing_cols}"
        )

    df["Date"] = pd.to_datetime(
        df["Date"],
        errors="raise",
    )

    for col in NUMERIC_REQUIRED_COLS:
        df[col] = pd.to_numeric(
            df[col],
            errors="raise",
        )

    df = (
        df
        .sort_values("Date")
        .reset_index(drop=True)
    )

    duplicate_dates = int(
        df["Date"].duplicated().sum()
    )

    if duplicate_dates > 0:
        raise ValueError(
            f"{split_name}: {duplicate_dates} duplicate dates detected."
        )

    missing_required = int(
        df[REQUIRED_COLS]
        .isna()
        .sum()
        .sum()
    )

    if missing_required > 0:
        raise ValueError(
            f"{split_name}: {missing_required} missing required values detected."
        )

    numeric_array = df[
        NUMERIC_REQUIRED_COLS
    ].to_numpy(dtype=float)

    nonfinite_numeric = int(
        (~np.isfinite(numeric_array)).sum()
    )

    if nonfinite_numeric > 0:
        raise ValueError(
            f"{split_name}: {nonfinite_numeric} non-finite numeric values detected."
        )

    # Robustness proxy only.
    df["return_squared_proxy"] = (
        df["fx_return"] ** 2
    )

    return df


def summarize_split(
    df: pd.DataFrame,
    split_name: str,
) -> dict:
    """Return an auditable summary for one chronological split."""
    return {
        "split": split_name,
        "n_rows": int(len(df)),
        "date_start": df["Date"].min(),
        "date_end": df["Date"].max(),
        "is_sorted_ascending": bool(
            df["Date"].is_monotonic_increasing
        ),
        "duplicate_dates": int(
            df["Date"].duplicated().sum()
        ),
        "missing_required_values": int(
            df[REQUIRED_COLS]
            .isna()
            .sum()
            .sum()
        ),
        "nonfinite_numeric_values": int(
            (
                ~np.isfinite(
                    df[NUMERIC_REQUIRED_COLS]
                    .to_numpy(dtype=float)
                )
            ).sum()
        ),
        "zero_fx_return_count": int(
            (df["fx_return"] == 0).sum()
        ),
        "zero_fx_return_share": float(
            (df["fx_return"] == 0).mean()
        ),
        "min_fx_return": float(
            df["fx_return"].min()
        ),
        "max_fx_return": float(
            df["fx_return"].max()
        ),
    }


def summarize_split_transition(
    earlier: pd.DataFrame,
    later: pd.DataFrame,
    transition_name: str,
) -> dict:
    """Audit chronology and date overlap between two adjacent splits."""
    overlap = set(
        earlier["Date"]
    ).intersection(
        set(later["Date"])
    )

    chronology_ok = bool(
        earlier["Date"].max()
        < later["Date"].min()
    )

    return {
        "transition": transition_name,
        "earlier_end": earlier["Date"].max(),
        "later_start": later["Date"].min(),
        "chronology_ok": chronology_ok,
        "overlap_date_count": int(len(overlap)),
    }


df_train = load_split(
    TRAIN_PATH,
    "train",
)

df_valid = load_split(
    VALID_PATH,
    "validation",
)

df_test = load_split(
    TEST_PATH,
    "test",
)


data_audit = pd.DataFrame([
    summarize_split(df_train, "train"),
    summarize_split(df_valid, "validation"),
    summarize_split(df_test, "test"),
])

split_chronology_audit = pd.DataFrame([
    summarize_split_transition(
        df_train,
        df_valid,
        "train_to_validation",
    ),
    summarize_split_transition(
        df_valid,
        df_test,
        "validation_to_test",
    ),
])

assert data_audit[
    "is_sorted_ascending"
].all(), "At least one split is not sorted chronologically."

assert (
    data_audit["duplicate_dates"] == 0
).all(), "Duplicate dates detected within at least one split."

assert (
    data_audit["missing_required_values"] == 0
).all(), "Missing required values detected."

assert (
    data_audit["nonfinite_numeric_values"] == 0
).all(), "Non-finite numeric values detected."

assert split_chronology_audit[
    "chronology_ok"
].all(), "Train, validation, and test splits are not strictly chronological."

assert (
    split_chronology_audit["overlap_date_count"] == 0
).all(), "Date overlap detected between adjacent splits."


data_audit.to_csv(
    OUTPUT_DIR / "table_00_data_audit.csv",
    index=False,
)

split_chronology_audit.to_csv(
    OUTPUT_DIR / "table_00b_split_chronology_audit.csv",
    index=False,
)


print("Data audit: PASSED")
print("Shock cols:", SHOCK_COLS)

display(data_audit)
display(split_chronology_audit)


Data audit: PASSED
Shock cols: ['vix_change_lag1', 'us10y_change_lag1', 'dxy_return_lag1', 'oil_change_lag1', 'vnindex_return_lag1']


,split,n_rows,date_start,date_end,is_sorted_ascending,duplicate_dates,missing_required_values,nonfinite_numeric_values,zero_fx_return_count,zero_fx_return_share,min_fx_return,max_fx_return
0,train,3014,2010-01-06,2021-12-31,True,0,0,0,465,0.154280,-2.394219,6.213884
1,validation,501,2022-01-03,2023-12-29,True,0,0,0,59,0.117764,-1.535608,1.197915
2,test,504,2024-01-02,2025-12-31,True,0,0,0,60,0.119048,-0.866927,0.881729


,transition,earlier_end,later_start,chronology_ok,overlap_date_count
0,train_to_validation,2021-12-31,2022-01-03,True,0
1,validation_to_test,2023-12-29,2024-01-02,True,0


In [3]:
# ============================================================
# 2. FORECAST METRICS AND STRICT VARIANCE VALIDATION
# ============================================================

def clip_positive_variance(
    values: Any,
    context: str = "variance",
) -> np.ndarray:
    """
    Apply a numerical floor only to strictly positive variances.

    Zero, negative, NaN, or infinite values are model failures.
    They must not be silently repaired.
    """
    arr = np.asarray(
        values,
        dtype=float,
    )

    if arr.size == 0:
        raise ValueError(
            f"{context}: empty variance array."
        )

    if not np.isfinite(arr).all():
        raise ValueError(
            f"{context}: non-finite variance detected."
        )

    if np.any(arr <= 0):
        raise ValueError(
            f"{context}: non-positive variance detected."
        )

    return np.maximum(
        arr,
        MIN_FORECAST_VAR,
    )


def qlike_loss(
    actual_var: Any,
    forecast_var: Any,
) -> np.ndarray:
    actual = np.asarray(
        actual_var,
        dtype=float,
    )

    if not np.isfinite(actual).all():
        raise ValueError(
            "QLIKE actual proxy contains non-finite values."
        )

    if np.any(actual < 0):
        raise ValueError(
            "QLIKE actual proxy contains negative values."
        )

    forecast = clip_positive_variance(
        forecast_var,
        context="QLIKE forecast variance",
    )

    return (
        np.log(
            forecast
        )
        + actual
        / forecast
    )


def evaluate_forecasts(
    predictions: pd.DataFrame,
    proxy_col: str = "innovation_variance_proxy",
    sample_name: str = "validation",
) -> pd.DataFrame:
    required = {
        "Date",
        "model",
        "forecast_var",
        proxy_col,
    }

    missing = sorted(
        required - set(predictions.columns)
    )

    if missing:
        raise ValueError(
            f"Forecast table is missing required columns: {missing}"
        )

    rows = []

    for model_name, group in predictions.groupby(
        "model",
        sort=False,
    ):
        d = (
            group
            .sort_values(
                "Date"
            )
            .copy()
        )

        actual = (
            d[
                proxy_col
            ]
            .to_numpy(
                dtype=float
            )
        )

        forecast = (
            d[
                "forecast_var"
            ]
            .to_numpy(
                dtype=float
            )
        )

        eligible = (
            len(
                d
            )
            > 0
            and np.isfinite(
                actual
            ).all()
            and np.isfinite(
                forecast
            ).all()
            and np.all(
                actual >= 0
            )
            and np.all(
                forecast > 0
            )
        )

        row = {
            "sample":
                sample_name,

            "model":
                model_name,

            "n_forecasts":
                len(
                    d
                ),

            "eligible_for_ranking":
                eligible,

            "QLIKE":
                np.nan,

            "MSE":
                np.nan,

            "RMSE":
                np.nan,

            "MAE":
                np.nan,

            "mean_forecast_variance":
                np.nan,

            "mean_actual_variance_proxy":
                np.nan,
        }

        if eligible:
            errors = (
                actual
                - forecast
            )

            row.update({
                "QLIKE":
                    float(
                        np.mean(
                            qlike_loss(
                                actual,
                                forecast,
                            )
                        )
                    ),

                "MSE":
                    float(
                        np.mean(
                            errors
                            ** 2
                        )
                    ),

                "RMSE":
                    float(
                        np.sqrt(
                            np.mean(
                                errors
                                ** 2
                            )
                        )
                    ),

                "MAE":
                    float(
                        np.mean(
                            np.abs(
                                errors
                            )
                        )
                    ),

                "mean_forecast_variance":
                    float(
                        np.mean(
                            forecast
                        )
                    ),

                "mean_actual_variance_proxy":
                    float(
                        np.mean(
                            actual
                        )
                    ),
            })

        rows.append(
            row
        )

    metrics = pd.DataFrame(
        rows
    )

    valid_mask = (
        metrics[
            "eligible_for_ranking"
        ]
    )

    metrics[
        "QLIKE_rank"
    ] = np.nan

    metrics.loc[
        valid_mask,
        "QLIKE_rank",
    ] = (
        metrics.loc[
            valid_mask,
            "QLIKE",
        ]
        .rank(
            method="min",
            ascending=True,
        )
    )

    return (
        metrics
        .sort_values(
            [
                "eligible_for_ranking",
                "QLIKE",
                "MSE",
            ],
            ascending=[
                False,
                True,
                True,
            ],
            na_position="last",
        )
        .reset_index(
            drop=True
        )
    )


In [4]:

# ============================================================
# 3. TRAIN-ONLY ARMA ORDER SELECTION WITH AUDIT TRAIL
# ============================================================

ARMA_MIN_LB_PVALUE = 0.05
MAX_ARMA_TOTAL_ORDER = 10


def fit_arma(y: pd.Series, p: int, q: int):
    """Fit a stationary and invertible ARMA(p, q) model."""
    return SARIMAX(
        y,
        order=(p, 0, q),
        trend="c",
        enforce_stationarity=True,
        enforce_invertibility=True,
    ).fit(
        disp=False,
        maxiter=1000,
    )


def build_arma_residual_frame(
    fitted_arma,
    source_frame: pd.DataFrame,
) -> pd.DataFrame:
    """Align ARMA innovations with dates and shock variables."""
    burn = int(fitted_arma.loglikelihood_burn)

    out = source_frame[
        ["Date", *SHOCK_COLS]
    ].copy()

    out["arma_resid"] = np.asarray(
        fitted_arma.resid,
        dtype=float,
    )

    return (
        out
        .iloc[burn:]
        .dropna(subset=["arma_resid", *SHOCK_COLS])
        .reset_index(drop=True)
    )


def empty_arma_audit_row(
    model_name: str,
    p: int,
    q: int,
) -> dict:
    """Create a complete ARMA audit row before fit details are known."""
    return {
        "model": model_name,
        "p": p,
        "q": q,
        "converged": False,
        "eligible_for_selection": False,
        "fit_status": "not_run",
        "error": "",
        "loglik": np.nan,
        "AIC": np.nan,
        "BIC": np.nan,
        "HQIC": np.nan,
        "num_params": np.nan,
        **{
            f"LB_resid_adj_p_lag{lag}": np.nan
            for lag in ARMA_LB_LAGS
        },
        "diagnostic_pass": False,
        "ARCH_LM_p_lag10": np.nan,
    }


def fit_arma_grid_train_only(
    train_frame: pd.DataFrame,
    max_p: int,
    max_q: int,
):
    """
    Select the ARMA mean equation using train data only.

    Failed and non-converged candidates remain in the exported table
    so the model search can be audited.
    """
    y = (
        train_frame["fx_return"]
        .astype(float)
        .reset_index(drop=True)
    )

    rows = []
    fits = {}

    for p in range(max_p + 1):
        for q in range(max_q + 1):

            # Avoid unnecessarily complex mean equations.
            if p + q > MAX_ARMA_TOTAL_ORDER:
                continue

            model_name = f"ARMA({p},{q})"
            row = empty_arma_audit_row(
                model_name=model_name,
                p=p,
                q=q,
            )

            try:
                result = fit_arma(
                    y=y,
                    p=p,
                    q=q,
                )

                row.update({
                    "loglik": float(result.llf),
                    "AIC": float(result.aic),
                    "BIC": float(result.bic),
                    "HQIC": float(result.hqic),
                    "num_params": int(len(result.params)),
                })

                converged = bool(
                    result.mle_retvals.get(
                        "converged",
                        False,
                    )
                )

                row["converged"] = converged

                if not converged:
                    row["fit_status"] = "optimizer_not_converged"
                    row["error"] = str(
                        result.mle_retvals
                    )[:300]
                    rows.append(row)
                    continue

                residual_frame = build_arma_residual_frame(
                    fitted_arma=result,
                    source_frame=train_frame,
                )

                residuals = residual_frame[
                    "arma_resid"
                ]

                lb_table = acorr_ljungbox(
                    residuals,
                    lags=ARMA_LB_LAGS,
                    model_df=p + q,
                    return_df=True,
                )

                lb_values = {
                    f"LB_resid_adj_p_lag{lag}":
                        float(
                            lb_table.loc[
                                lag,
                                "lb_pvalue",
                            ]
                        )
                    for lag in ARMA_LB_LAGS
                }

                diagnostic_pass = all(
                    lb_values[
                        f"LB_resid_adj_p_lag{lag}"
                    ]
                    >= ARMA_MIN_LB_PVALUE
                    for lag in ARMA_LB_LAGS
                )

                row.update({
                    **lb_values,
                    "diagnostic_pass": bool(
                        diagnostic_pass
                    ),
                    "ARCH_LM_p_lag10": float(
                        het_arch(
                            residuals,
                            nlags=10,
                            ddof=p + q,
                        )[1]
                    ),
                    "eligible_for_selection": True,
                    "fit_status": "ok",
                    "error": "",
                })

                fits[model_name] = result
                rows.append(row)

            except Exception as exc:
                row["fit_status"] = "exception"
                row["error"] = str(exc)[:300]
                rows.append(row)

    table = (
        pd.DataFrame(rows)
        .sort_values(
            [
                "eligible_for_selection",
                "diagnostic_pass",
                "BIC",
                "AIC",
                "num_params",
            ],
            ascending=[
                False,
                False,
                True,
                True,
                True,
            ],
            na_position="last",
        )
        .reset_index(drop=True)
    )

    return table, fits


# ------------------------------------------------------------
# Run train-only ARMA grid
# ------------------------------------------------------------

arma_table, arma_fits = fit_arma_grid_train_only(
    train_frame=df_train,
    max_p=MAX_ARMA_P,
    max_q=MAX_ARMA_Q,
)

eligible_arma = arma_table.loc[
    arma_table["eligible_for_selection"]
].copy()

if eligible_arma.empty:
    raise RuntimeError(
        "No converged ARMA candidate with computable diagnostics was found."
    )


# ------------------------------------------------------------
# Lock the mean equation before volatility model selection
# ------------------------------------------------------------

clean_arma = eligible_arma.loc[
    eligible_arma["diagnostic_pass"]
].copy()

if clean_arma.empty:
    warnings.warn(
        "No ARMA candidate passed adjusted Ljung-Box diagnostics. "
        "The lowest-BIC eligible model is used. "
        "Report remaining mean autocorrelation as a limitation."
    )

    arma_selection_pool = eligible_arma
    arma_selection_status = "fallback_lowest_BIC_eligible"

else:
    arma_selection_pool = clean_arma
    arma_selection_status = "lowest_BIC_among_diagnostic_pass_models"


best_arma_row = (
    arma_selection_pool
    .sort_values(["BIC", "AIC", "num_params"])
    .iloc[0]
)

LOCKED_ARMA_P = int(best_arma_row["p"])
LOCKED_ARMA_Q = int(best_arma_row["q"])
LOCKED_ARMA_NAME = str(best_arma_row["model"])

best_arma_fit_train = arma_fits[
    LOCKED_ARMA_NAME
]

train_arma_residual_frame = build_arma_residual_frame(
    fitted_arma=best_arma_fit_train,
    source_frame=df_train,
)

arma_table.to_csv(
    OUTPUT_DIR / "table_01_train_arma_selection.csv",
    index=False,
)

print("Locked ARMA order :", LOCKED_ARMA_NAME)
print("Selection status  :", arma_selection_status)
print("Train innovations :", len(train_arma_residual_frame))

display(arma_table)


Locked ARMA order : ARMA(4,5)
Selection status  : lowest_BIC_among_diagnostic_pass_models
Train innovations : 3014


,model,p,q,converged,eligible_for_selection,fit_status,error,loglik,AIC,BIC,HQIC,num_params,LB_resid_adj_p_lag10,LB_resid_adj_p_lag20,LB_resid_adj_p_lag30,diagnostic_pass,ARCH_LM_p_lag10
0,"ARMA(4,5)",4,5,True,True,ok,,349.260459,-676.520917,-610.399660,-652.742910,11,0.071936,0.058441,0.197838,True,0.001735
1,"ARMA(2,3)",2,3,True,True,ok,,339.002351,-664.004702,-621.927539,-648.873243,7,0.155939,0.026277,0.159098,False,0.001785
2,"ARMA(3,2)",3,2,True,True,ok,,338.495855,-662.991711,-620.914547,-647.860251,7,0.087536,0.021660,0.147695,False,0.002184
3,"ARMA(3,3)",3,3,True,True,ok,,339.161957,-662.323915,-614.235728,-645.030819,8,0.078999,0.017245,0.123191,False,0.001809
4,"ARMA(2,4)",2,4,True,True,ok,,339.122044,-662.244089,-614.155902,-644.950993,8,0.079987,0.016926,0.121795,False,0.001793
5,"ARMA(4,2)",4,2,True,True,ok,,339.040691,-662.081382,-613.993195,-644.788286,8,0.065778,0.015331,0.112633,False,0.001846
6,"ARMA(5,4)",5,4,True,True,ok,,349.524983,-677.049966,-610.928709,-653.271959,11,0.027396,0.061883,0.230982,False,0.002387
7,"ARMA(2,5)",2,5,True,True,ok,,339.992021,-661.984042,-607.884832,-642.529309,9,0.111257,0.031343,0.208787,False,0.002048
8,"ARMA(4,3)",4,3,True,True,ok,,339.417381,-660.834762,-606.735551,-641.380029,9,0.031707,0.013236,0.119789,False,0.002151
9,"ARMA(3,4)",3,4,True,True,ok,,339.234771,-660.469542,-606.370331,-641.014809,9,0.048972,0.013037,0.110464,False,0.001844


In [5]:
# ============================================================
# CAUSAL ONE-STEP-AHEAD ARMA INNOVATIONS
# ============================================================

def build_recursive_arma_innovations(
    fitted_arma,
    evaluation_frame: pd.DataFrame,
) -> pd.DataFrame:
    out = (
        evaluation_frame[
            ["Date", "fx_return", *SHOCK_COLS]
        ]
        .copy()
        .sort_values("Date")
        .reset_index(drop=True)
    )

    evaluation_result = fitted_arma.extend(
        out["fx_return"].astype(float).to_numpy()
    )

    out["arma_mean_forecast"] = np.asarray(
        evaluation_result.fittedvalues,
        dtype=float,
    ).reshape(-1)

    out["arma_innovation"] = np.asarray(
        evaluation_result.resid,
        dtype=float,
    ).reshape(-1)

    # Main volatility-evaluation proxy
    out["innovation_variance_proxy"] = (
        out["arma_innovation"] ** 2
    )

    # Robustness proxy
    out["return_squared_proxy"] = (
        out["fx_return"] ** 2
    )

    return out


valid_arma_output = build_recursive_arma_innovations(
    fitted_arma=best_arma_fit_train,
    evaluation_frame=df_valid,
)

display(valid_arma_output)

,Date,fx_return,vix_change_lag1,us10y_change_lag1,dxy_return_lag1,oil_change_lag1,vnindex_return_lag1,arma_mean_forecast,arma_innovation,innovation_variance_proxy,return_squared_proxy
0,2022-01-03,0.000000,-0.110000,0.000000,-0.313090,-1.500000,0.825003,-0.040124,0.040124,0.001610,0.000000
1,2022-01-04,-0.444724,-0.620000,0.110000,0.594030,0.660000,0.000000,-0.039031,-0.405693,0.164587,0.197779
2,2022-01-05,0.034139,0.310000,0.030000,0.051943,1.010000,1.805688,0.120519,-0.086380,0.007461,0.001165
3,2022-01-06,-0.045517,2.820000,0.050000,-0.103905,0.830000,-0.202095,-0.014553,-0.030964,0.000959,0.002072
4,2022-01-07,-0.272665,-0.120000,0.020000,0.062355,1.640000,0.397894,0.012232,-0.284897,0.081166,0.074346
...,...,...,...,...,...,...,...,...,...,...,...
496,2023-12-22,-0.340095,-0.020000,0.030000,-0.558148,-0.280000,0.151598,0.027042,-0.367137,0.134789,0.115665
497,2023-12-26,0.364432,-0.620000,0.010000,-0.137565,-0.300000,0.057130,0.097258,0.267174,0.071382,0.132811
498,2023-12-27,0.097407,-0.040000,-0.010000,-0.226407,2.550000,1.724746,-0.025005,0.122412,0.014985,0.009488
499,2023-12-28,-0.461838,-0.560000,-0.100000,-0.474172,-1.530000,-0.023170,-0.060195,-0.401643,0.161317,0.213295


In [6]:
# ============================================================
# VOLATILITY MODELS: FIT AND RECURSIVE FORECAST
# ============================================================

from collections import deque
from scipy.optimize import minimize
from scipy.special import gammaln
from arch import arch_model

SQRT_2_OVER_PI = float(np.sqrt(2.0 / np.pi))


# ------------------------------------------------------------
# 5.1. Shared helpers
# ------------------------------------------------------------

def clean_residuals(
    values,
    context: str = "ARMA innovations",
) -> np.ndarray:
    """
    Return a strict residual array.

    Non-finite residuals are model failures and must not be dropped
    silently because dropping them can misalign recursive forecasts.
    """
    eps = np.asarray(values, dtype=float)

    if eps.size == 0:
        raise ValueError(f"{context}: empty residual array.")

    if not np.isfinite(eps).all():
        raise ValueError(
            f"{context}: non-finite residual detected."
        )

    return eps


def attach_forecast(
    evaluation: pd.DataFrame,
    forecast_var,
    model_name: str,
) -> pd.DataFrame:
    """Attach variance forecasts to evaluation-period innovations."""
    forecast_var = np.asarray(forecast_var, dtype=float)

    if len(evaluation) != len(forecast_var):
        raise ValueError("Forecast length does not match evaluation sample.")

    out = evaluation[
        [
            "Date",
            "fx_return",
            "arma_mean_forecast",
            "arma_innovation",
            "innovation_variance_proxy",
            "return_squared_proxy",
        ]
    ].copy()

    out["model"] = model_name
    out["forecast_var"] = clip_positive_variance(
        forecast_var,
        context=f"{model_name}: recursive forecast variance",
    )

    return out


def fit_shock_scaler(frame: pd.DataFrame) -> dict[str, np.ndarray]:
    """Fit shock standardization parameters on the estimation sample only."""
    mean = frame[SHOCK_COLS].mean().to_numpy(dtype=float)
    std = frame[SHOCK_COLS].std(ddof=0).to_numpy(dtype=float)
    std = np.where(std > 0, std, 1.0)

    return {
        "mean": mean,
        "std": std,
    }


def scale_shocks(
    frame: pd.DataFrame,
    scaler: dict[str, np.ndarray],
    use_absolute_values: bool,
) -> np.ndarray:
    """Standardize shocks using estimation-sample statistics."""
    x = (
        frame[SHOCK_COLS].to_numpy(dtype=float)
        - scaler["mean"]
    ) / scaler["std"]

    return np.abs(x) if use_absolute_values else x


def backcast_variance(residuals: np.ndarray) -> float:
    """EWMA backcast used to initialize custom variance recursion."""
    tau = min(75, len(residuals))
    weights = 0.94 ** np.arange(tau)
    weights /= weights.sum()

    return max(
        float(weights @ (residuals[:tau] ** 2)),
        MIN_FORECAST_VAR,
    )


# ------------------------------------------------------------
# 5.2. Standard GARCH-family models
# ------------------------------------------------------------

STANDARD_MODEL_SPECS = {
    "GARCH(1,1)-t": {
        "vol": "GARCH",
        "p": 1,
        "o": 0,
        "q": 1,
    },
    "GJR-GARCH(1,1)-t": {
        "vol": "GARCH",
        "p": 1,
        "o": 1,
        "q": 1,
    },
    "EGARCH(1,1)-t": {
        "vol": "EGARCH",
        "p": 1,
        "o": 1,
        "q": 1,
    },
}


def fit_standard_volatility_model(
    model_name: str,
    train_residuals,
    dist: str = "t"
) -> dict:
    """Fit a GARCH-family model with the requested innovation distribution."""
    eps = clean_residuals(
        train_residuals,
        context=f"{model_name}: estimation residuals",
    )

    result = arch_model(
        eps,
        mean="Zero",
        dist=dist,
        rescale=False,
        **STANDARD_MODEL_SPECS[model_name],
    ).fit(
        disp="off",
        update_freq=0,
        show_warning=False,
    )

    conditional_var = clip_positive_variance(
        np.asarray(
            result.conditional_volatility,
            dtype=float,
        ) ** 2,
        context=f"{model_name}: in-sample conditional variance",
    )

    return {
        "kind": "standard",
        "model_name": model_name,
        "params": result.params.copy(),
        "fit_result": result,
        "last_eps": float(eps[-1]),
        "last_h": float(conditional_var[-1]),
        "loglik": float(result.loglikelihood),
        "AIC": float(result.aic),
        "BIC": float(result.bic),
        "converged": int(result.convergence_flag) == 0,
        "in_sample_residuals": eps.copy(),
        "in_sample_variance": conditional_var.copy(),
    }


def standard_one_step_variance(
    fitted_model: dict,
    previous_eps: float,
    previous_h: float,
) -> float:
    """
    Compute the next conditional variance.

    The distribution affects fitted parameters but does not change
    the variance-recursion identity.
    """
    name = fitted_model.get(
        "variance_specification",
        fitted_model["model_name"],
    )

    params = fitted_model["params"]

    omega = float(params["omega"])
    alpha = float(params["alpha[1]"])
    beta = float(params["beta[1]"])

    if name == "GARCH(1,1)-t":
        h = (
            omega
            + alpha * previous_eps**2
            + beta * previous_h
        )

    elif name == "GJR-GARCH(1,1)-t":
        gamma = float(params["gamma[1]"])

        h = (
            omega
            + alpha * previous_eps**2
            + gamma * float(previous_eps < 0) * previous_eps**2
            + beta * previous_h
        )

    elif name == "EGARCH(1,1)-t":
        gamma = float(params["gamma[1]"])

        z = previous_eps / np.sqrt(
            max(previous_h, MIN_FORECAST_VAR)
        )

        log_h = (
            omega
            + alpha * (abs(z) - SQRT_2_OVER_PI)
            + gamma * z
            + beta * np.log(
                max(previous_h, MIN_FORECAST_VAR)
            )
        )

        h = np.exp(
            np.clip(
                log_h,
                np.log(MIN_FORECAST_VAR),
                20.0,
            )
        )

    else:
        raise ValueError(
            f"Unsupported variance specification: {name}"
        )

    return float(
        clip_positive_variance(
            [h],
            context=f"{name}: one-step forecast variance",
        )[0]
    )


def forecast_standard_volatility(
    fitted_model: dict,
    evaluation: pd.DataFrame,
) -> pd.DataFrame:
    """Generate recursive one-step-ahead forecasts with fixed parameters."""
    previous_eps = fitted_model["last_eps"]
    previous_h = fitted_model["last_h"]
    forecasts = []

    for row in evaluation.itertuples(index=False):
        previous_h = standard_one_step_variance(
            fitted_model=fitted_model,
            previous_eps=previous_eps,
            previous_h=previous_h,
        )

        forecasts.append(previous_h)
        previous_eps = float(row.arma_innovation)

    return attach_forecast(
        evaluation=evaluation,
        forecast_var=forecasts,
        model_name=fitted_model["model_name"],
    )


# ------------------------------------------------------------
# 5.3. Custom EGARCH and EGARCH-X models
# ------------------------------------------------------------

def egarchx_variance_path(
    params: np.ndarray,
    residuals: np.ndarray,
    x: np.ndarray,
) -> np.ndarray:
    """
    Compute the in-sample custom EGARCH variance path.

    The same function is used for:
    - Custom EGARCH no-X-t
    - EGARCH-X signed lag1-t
    - EGARCH-X abs lag1-t

    For the no-X model, x has zero columns and delta @ x[t] equals 0.
    """
    omega, alpha, gamma, beta = params[:4]
    delta = params[4:-1]

    h = np.empty(len(residuals), dtype=float)
    h[0] = backcast_variance(residuals)

    for t in range(1, len(residuals)):
        z_previous = residuals[t - 1] / np.sqrt(
            max(h[t - 1], MIN_FORECAST_VAR)
        )

        shock_term = float(delta @ x[t])

        log_h = (
            omega
            + alpha * (abs(z_previous) - SQRT_2_OVER_PI)
            + gamma * z_previous
            + beta * np.log(
                max(h[t - 1], MIN_FORECAST_VAR)
            )
            + shock_term
        )

        h[t] = np.exp(
            np.clip(
                log_h,
                np.log(MIN_FORECAST_VAR),
                20.0,
            )
        )

    return clip_positive_variance(
        h,
        context="Custom EGARCH in-sample conditional variance",
    )


def standardized_student_t_loglik(
    residuals: np.ndarray,
    h: np.ndarray,
    nu: float,
) -> float:
    """Student-t log likelihood standardized to unit variance."""
    if nu <= 2.0 or np.any(h <= 0) or not np.isfinite(h).all():
        return -np.inf

    log_density = (
        gammaln((nu + 1.0) / 2.0)
        - gammaln(nu / 2.0)
        - 0.5 * np.log(np.pi * (nu - 2.0))
        - 0.5 * np.log(h)
        - ((nu + 1.0) / 2.0)
        * np.log1p(
            residuals**2
            / (h * (nu - 2.0))
        )
    )

    return float(log_density.sum())


def fit_egarchx_model(
    model_name: str,
    train_residual_frame: pd.DataFrame,
    use_absolute_values: bool,
    include_shocks: bool = True,
) -> dict:
    """
    Fit a custom EGARCH or EGARCH-X model by manual maximum likelihood.

    include_shocks=False:
        Fit Custom EGARCH no-X-t.

    include_shocks=True:
        Fit signed-shock or magnitude-shock EGARCH-X.

    The no-X model retains the same estimation sample as the EGARCH-X
    models so that forecast comparisons are directly comparable.
    """
    frame = (
        train_residual_frame[
            ["Date", "arma_resid", *SHOCK_COLS]
        ]
        .dropna()
        .reset_index(drop=True)
    )

    residuals = clean_residuals(
        frame["arma_resid"].to_numpy(dtype=float),
        context=f"{model_name}: estimation residuals",
    )

    if include_shocks:
        scaler = fit_shock_scaler(frame)

        x = scale_shocks(
            frame=frame,
            scaler=scaler,
            use_absolute_values=use_absolute_values,
        )

        feature_names = list(SHOCK_COLS)

    else:
        scaler = {
            "mean": np.empty(0, dtype=float),
            "std": np.empty(0, dtype=float),
        }

        x = np.empty(
            (len(frame), 0),
            dtype=float,
        )

        feature_names = []

    n_features = x.shape[1]

    def objective(params: np.ndarray) -> float:
        h = egarchx_variance_path(
            params=params,
            residuals=residuals,
            x=x,
        )

        loglik = standardized_student_t_loglik(
            residuals=residuals,
            h=h,
            nu=float(params[-1]),
        )

        return -loglik if np.isfinite(loglik) else 1e20

    variance = max(
        float(np.var(residuals, ddof=1)),
        MIN_FORECAST_VAR,
    )

    # EGARCH does not require GARCH-style positivity constraints.
    # Stability is imposed through abs(beta) < 1.
    bounds = [
        (-10.0, 10.0),               # omega
        (-5.0, 5.0),                 # alpha
        (-5.0, 5.0),                 # gamma
        (-0.999, 0.999),             # beta
        *[(-5.0, 5.0)] * n_features, # delta
        (2.05, 100.0),               # nu
    ]

    starts = []

    for beta_start in [0.85, 0.90, 0.95, 0.98]:
        starts.append(
            np.array(
                [
                    np.log(variance) * (1.0 - beta_start),
                    0.10,
                    0.00,
                    beta_start,
                    *([0.0] * n_features),
                    8.0,
                ],
                dtype=float,
            )
        )

    results = [
        minimize(
            objective,
            x0=start,
            method="L-BFGS-B",
            bounds=bounds,
            options={"maxiter": 3000},
        )
        for start in starts
    ]

    valid_results = [
        result
        for result in results
        if result.success and np.isfinite(result.fun)
    ]

    # Derivative-free fallback for line-search failures.
    if not valid_results:
        seed = min(
            results,
            key=lambda result: (
                float(result.fun)
                if np.isfinite(result.fun)
                else np.inf
            ),
        ).x

        fallback = minimize(
            objective,
            x0=seed,
            method="Powell",
            bounds=bounds,
            options={"maxiter": 3000},
        )

        if fallback.success and np.isfinite(fallback.fun):
            valid_results = [fallback]

    if not valid_results:
        raise RuntimeError(f"{model_name} failed to converge.")

    best_result = min(
        valid_results,
        key=lambda result: float(result.fun),
    )

    params = np.asarray(
        best_result.x,
        dtype=float,
    )

    h = egarchx_variance_path(
        params=params,
        residuals=residuals,
        x=x,
    )

    loglik = -float(best_result.fun)
    n_params = len(params)
    n_obs = len(residuals)

    parameter_names = [
        "omega",
        "alpha",
        "gamma",
        "beta",
        *[
            f"delta_{col}"
            for col in feature_names
        ],
        "nu",
    ]

    return {
        "kind": "custom_egarchx",
        "model_name": model_name,
        "include_shocks": include_shocks,
        "feature_names": feature_names,
        "params": pd.Series(
            params,
            index=parameter_names,
        ),
        "scaler": scaler,
        "use_absolute_values": use_absolute_values,
        "last_eps": float(residuals[-1]),
        "last_h": float(h[-1]),
        "loglik": loglik,
        "AIC": float(
            -2.0 * loglik
            + 2.0 * n_params
        ),
        "BIC": float(
            -2.0 * loglik
            + np.log(n_obs) * n_params
        ),
        "converged": True,
        "in_sample_residuals": residuals.copy(),
        "in_sample_variance": h.copy(),
    }


def custom_egarchx_one_step_variance(
    fitted_model: dict,
    previous_eps: float,
    previous_h: float,
    current_x_raw: np.ndarray,
) -> float:
    """
    Compute one-step-ahead custom EGARCH variance.

    For Custom EGARCH no-X-t, shock_term is fixed at zero.
    """
    params = fitted_model["params"]

    if fitted_model["include_shocks"]:
        scaler = fitted_model["scaler"]

        x = (
            np.asarray(
                current_x_raw,
                dtype=float,
            )
            - scaler["mean"]
        ) / scaler["std"]

        if fitted_model["use_absolute_values"]:
            x = np.abs(x)

        delta = params[
            [
                f"delta_{col}"
                for col in fitted_model["feature_names"]
            ]
        ].to_numpy(dtype=float)

        shock_term = float(delta @ x)

    else:
        shock_term = 0.0

    z_previous = previous_eps / np.sqrt(
        max(previous_h, MIN_FORECAST_VAR)
    )

    log_h = (
        float(params["omega"])
        + float(params["alpha"])
        * (abs(z_previous) - SQRT_2_OVER_PI)
        + float(params["gamma"])
        * z_previous
        + float(params["beta"])
        * np.log(
            max(previous_h, MIN_FORECAST_VAR)
        )
        + shock_term
    )

    return float(
        np.exp(
            np.clip(
                log_h,
                np.log(MIN_FORECAST_VAR),
                20.0,
            )
        )
    )


def forecast_custom_egarchx(
    fitted_model: dict,
    evaluation: pd.DataFrame,
) -> pd.DataFrame:
    """
    Generate recursive one-step-ahead forecasts for custom EGARCH models.
    """
    previous_eps = fitted_model["last_eps"]
    previous_h = fitted_model["last_h"]
    forecasts = []

    for row in evaluation.itertuples(index=False):
        if fitted_model["include_shocks"]:
            x_raw = np.array(
                [
                    getattr(row, col)
                    for col in fitted_model["feature_names"]
                ],
                dtype=float,
            )

        else:
            x_raw = np.empty(
                0,
                dtype=float,
            )

        previous_h = custom_egarchx_one_step_variance(
            fitted_model=fitted_model,
            previous_eps=previous_eps,
            previous_h=previous_h,
            current_x_raw=x_raw,
        )

        forecasts.append(previous_h)
        previous_eps = float(
            row.arma_innovation
        )

    return attach_forecast(
        evaluation=evaluation,
        forecast_var=forecasts,
        model_name=fitted_model["model_name"],
    )

# ------------------------------------------------------------
# 5.4. Simple benchmarks
# ------------------------------------------------------------

def forecast_ewma(
    train_residuals,
    evaluation: pd.DataFrame,
    lambda_: float = EWMA_LAMBDA,
) -> pd.DataFrame:
    """
    Recursive EWMA benchmark.

    The estimation residual history is filtered first.
    The resulting end-of-sample variance is then used
    as the first one-step-ahead forecast.
    """
    eps = clean_residuals(
        train_residuals,
        context="EWMA benchmark estimation residuals",
    )

    previous_h = float(
        np.mean(eps**2)
    )

    # Filter through the full estimation history.
    for residual in eps:
        previous_h = max(
            lambda_ * previous_h
            + (1.0 - lambda_) * float(residual) ** 2,
            MIN_FORECAST_VAR,
        )

    forecasts = []

    # Fixed-parameter recursive one-step-ahead forecasts.
    for row in evaluation.itertuples(index=False):
        forecasts.append(
            previous_h
        )

        previous_h = max(
            lambda_ * previous_h
            + (1.0 - lambda_)
            * float(row.arma_innovation) ** 2,
            MIN_FORECAST_VAR,
        )

    return attach_forecast(
        evaluation=evaluation,
        forecast_var=forecasts,
        model_name=f"EWMA_{lambda_:.2f}",
    )


def forecast_historical_variance(
    train_residuals,
    evaluation: pd.DataFrame,
) -> pd.DataFrame:
    """Expanding historical-variance benchmark."""
    eps = clean_residuals(
        train_residuals,
        context="HistoricalVariance benchmark estimation residuals",
    )

    sum_squared = float(np.sum(eps**2))
    n_obs = len(eps)
    forecasts = []

    for row in evaluation.itertuples(index=False):
        forecasts.append(
            max(sum_squared / n_obs, MIN_FORECAST_VAR)
        )

        sum_squared += float(row.arma_innovation) ** 2
        n_obs += 1

    return attach_forecast(
        evaluation=evaluation,
        forecast_var=forecasts,
        model_name="HistoricalVariance",
    )


def forecast_rolling_variance(
    train_residuals,
    evaluation: pd.DataFrame,
    window: int = 22,
) -> pd.DataFrame:
    """Rolling 22-day variance benchmark."""
    eps = clean_residuals(
        train_residuals,
        context=f"RollingVariance{window} benchmark estimation residuals",
    )
    history = deque(eps[-window:], maxlen=window)
    forecasts = []

    for row in evaluation.itertuples(index=False):
        forecasts.append(
            max(
                float(np.mean(np.square(history))),
                MIN_FORECAST_VAR,
            )
        )

        history.append(float(row.arma_innovation))

    return attach_forecast(
        evaluation=evaluation,
        forecast_var=forecasts,
        model_name=f"RollingVariance{window}",
    )


# ------------------------------------------------------------
# 5.5. Dispatchers
# ------------------------------------------------------------

def fit_parametric_model(
    model_name: str,
    train_residual_frame: pd.DataFrame,
) -> dict:
    """Fit one parametric volatility model."""
    if model_name in STANDARD_MODEL_SPECS:
        fitted = fit_standard_volatility_model(
            model_name=model_name,
            train_residuals=train_residual_frame["arma_resid"],
        )

    elif model_name == "Custom EGARCH no-X-t":
        fitted = fit_egarchx_model(
            model_name=model_name,
            train_residual_frame=train_residual_frame,
            use_absolute_values=False,
            include_shocks=False,
        )

    elif model_name == "EGARCH-X signed lag1-t":
        fitted = fit_egarchx_model(
            model_name=model_name,
            train_residual_frame=train_residual_frame,
            use_absolute_values=False,
            include_shocks=True,
        )

    elif model_name == "EGARCH-X abs lag1-t":
        fitted = fit_egarchx_model(
            model_name=model_name,
            train_residual_frame=train_residual_frame,
            use_absolute_values=True,
            include_shocks=True,
        )

    else:
        raise ValueError(
            f"Unsupported parametric model: {model_name}"
        )

    if not fitted["converged"]:
        raise RuntimeError(
            f"{model_name} failed to converge."
        )

    return fitted


def forecast_parametric_model(
    fitted_model: dict,
    evaluation: pd.DataFrame,
) -> pd.DataFrame:
    """Forecast one fitted parametric volatility model."""
    if fitted_model["kind"] == "standard":
        return forecast_standard_volatility(
            fitted_model=fitted_model,
            evaluation=evaluation,
        )

    if fitted_model["kind"] == "custom_egarchx":
        return forecast_custom_egarchx(
            fitted_model=fitted_model,
            evaluation=evaluation,
        )

    raise ValueError(
        f"Unsupported fitted-model kind: {fitted_model['kind']}"
    )

# ------------------------------------------------------------
# 5.6. Residual diagnostics after volatility fit
# ------------------------------------------------------------

VOLATILITY_DIAGNOSTIC_LAGS = [
    10,
    20,
]

VOLATILITY_ARCH_LAGS = 10


def failed_volatility_diagnostic_row(
    model_name: str,
    estimation_sample: str,
    error: str,
) -> dict:
    """Return a complete failed-diagnostics row for auditability."""
    return {
        "estimation_sample": estimation_sample,
        "model": model_name,
        "n_std_resid": np.nan,
        "mean_std_resid": np.nan,
        "std_std_resid": np.nan,
        "max_abs_std_resid": np.nan,
        "n_abs_std_resid_gt3": np.nan,
        **{
            f"LB_std_resid_p_lag{lag}": np.nan
            for lag in VOLATILITY_DIAGNOSTIC_LAGS
        },
        **{
            f"LB_squared_std_resid_p_lag{lag}": np.nan
            for lag in VOLATILITY_DIAGNOSTIC_LAGS
        },
        "ARCH_LM_p_lag10": np.nan,
        "diagnostic_pass": False,
        "diagnostic_error": str(error)[:300],
    }


def volatility_residual_diagnostics(
    fitted_model: dict,
    estimation_sample: str,
) -> dict:
    """
    Evaluate standardized residuals after each volatility fit.

    A model passes only when:
    - standardized residuals have no Ljung-Box rejection at lags 10 and 20;
    - squared standardized residuals have no Ljung-Box rejection at lags 10 and 20;
    - ARCH-LM does not reject at 5 percent.
    """
    model_name = str(
        fitted_model["model_name"]
    )

    residuals = clean_residuals(
        fitted_model["in_sample_residuals"],
        context=f"{model_name}: diagnostic residuals",
    )

    variance = clip_positive_variance(
        fitted_model["in_sample_variance"],
        context=f"{model_name}: diagnostic conditional variance",
    )

    if len(residuals) != len(variance):
        raise ValueError(
            f"{model_name}: diagnostic residual and variance lengths differ."
        )

    std_resid = residuals / np.sqrt(
        variance
    )

    if not np.isfinite(std_resid).all():
        raise ValueError(
            f"{model_name}: non-finite standardized residual detected."
        )

    lb_std = acorr_ljungbox(
        std_resid,
        lags=VOLATILITY_DIAGNOSTIC_LAGS,
        return_df=True,
    )

    lb_squared = acorr_ljungbox(
        std_resid**2,
        lags=VOLATILITY_DIAGNOSTIC_LAGS,
        return_df=True,
    )

    arch_lm_pvalue = float(
        het_arch(
            std_resid,
            nlags=VOLATILITY_ARCH_LAGS,
        )[1]
    )

    lb_std_values = {
        f"LB_std_resid_p_lag{lag}":
            float(
                lb_std.loc[
                    lag,
                    "lb_pvalue",
                ]
            )
        for lag in VOLATILITY_DIAGNOSTIC_LAGS
    }

    lb_squared_values = {
        f"LB_squared_std_resid_p_lag{lag}":
            float(
                lb_squared.loc[
                    lag,
                    "lb_pvalue",
                ]
            )
        for lag in VOLATILITY_DIAGNOSTIC_LAGS
    }

    diagnostic_pass = (
        all(
            value > 0.05
            for value in lb_std_values.values()
        )
        and all(
            value > 0.05
            for value in lb_squared_values.values()
        )
        and arch_lm_pvalue > 0.05
    )

    return {
        "estimation_sample": estimation_sample,
        "model": model_name,
        "n_std_resid": int(
            len(std_resid)
        ),
        "mean_std_resid": float(
            np.mean(std_resid)
        ),
        "std_std_resid": float(
            np.std(
                std_resid,
                ddof=1,
            )
        ),
        "max_abs_std_resid": float(
            np.max(
                np.abs(std_resid)
            )
        ),
        "n_abs_std_resid_gt3": int(
            (
                np.abs(std_resid)
                > 3.0
            ).sum()
        ),
        **lb_std_values,
        **lb_squared_values,
        "ARCH_LM_p_lag10": arch_lm_pvalue,
        "diagnostic_pass": bool(
            diagnostic_pass
        ),
        "diagnostic_error": "",
    }


In [7]:
# ============================================================
# 6. TRAIN FIT -> RESIDUAL DIAGNOSTICS -> VALIDATION LOCK
# ============================================================

PARAMETRIC_MODELS = [
    "GARCH(1,1)-t",
    "GJR-GARCH(1,1)-t",
    "EGARCH(1,1)-t",
    "Custom EGARCH no-X-t",
    "EGARCH-X signed lag1-t",
    "EGARCH-X abs lag1-t",
]


# ------------------------------------------------------------
# 6.1. Fit parametric models on train only
# ------------------------------------------------------------

train_fitted_models = {}
fit_rows = []
parameter_rows = []
scaler_rows = []
train_diagnostic_rows = []

for model_name in PARAMETRIC_MODELS:
    print("Training volatility fit |", model_name)

    try:
        fitted = fit_parametric_model(
            model_name=model_name,
            train_residual_frame=train_arma_residual_frame,
        )

        train_fitted_models[model_name] = fitted

        fit_rows.append({
            "model": model_name,
            "converged": bool(
                fitted["converged"]
            ),
            "loglik": float(
                fitted["loglik"]
            ),
            "AIC": float(
                fitted["AIC"]
            ),
            "BIC": float(
                fitted["BIC"]
            ),
            "error": "",
        })

        for parameter, estimate in fitted["params"].items():
            parameter_rows.append({
                "model": model_name,
                "parameter": parameter,
                "estimate": float(estimate),
            })

        if fitted["kind"] == "custom_egarchx":
            for feature, mean, std in zip(
                SHOCK_COLS,
                fitted["scaler"]["mean"],
                fitted["scaler"]["std"],
            ):
                scaler_rows.append({
                    "model": model_name,
                    "feature": feature,
                    "mean": float(mean),
                    "std": float(std),
                })

        try:
            train_diagnostic_rows.append(
                volatility_residual_diagnostics(
                    fitted_model=fitted,
                    estimation_sample="train",
                )
            )

        except Exception as diagnostic_exc:
            train_diagnostic_rows.append(
                failed_volatility_diagnostic_row(
                    model_name=model_name,
                    estimation_sample="train",
                    error=str(diagnostic_exc),
                )
            )

    except Exception as exc:
        fit_rows.append({
            "model": model_name,
            "converged": False,
            "loglik": np.nan,
            "AIC": np.nan,
            "BIC": np.nan,
            "error": str(exc)[:300],
        })

        train_diagnostic_rows.append(
            failed_volatility_diagnostic_row(
                model_name=model_name,
                estimation_sample="train",
                error=str(exc),
            )
        )


train_volatility_fit = (
    pd.DataFrame(fit_rows)
    .sort_values(
        ["converged", "BIC"],
        ascending=[False, True],
        na_position="last",
    )
    .reset_index(drop=True)
)

train_volatility_parameters = pd.DataFrame(
    parameter_rows
)

train_feature_scalers = pd.DataFrame(
    scaler_rows
)

train_volatility_residual_diagnostics = (
    pd.DataFrame(
        train_diagnostic_rows
    )
    .sort_values(
        [
            "diagnostic_pass",
            "model",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)


train_volatility_fit.to_csv(
    OUTPUT_DIR / "table_02_train_volatility_fit.csv",
    index=False,
)

train_volatility_residual_diagnostics.to_csv(
    OUTPUT_DIR
    / "table_02b_train_volatility_residual_diagnostics.csv",
    index=False,
)

train_volatility_parameters.to_csv(
    OUTPUT_DIR / "table_03_train_volatility_parameters.csv",
    index=False,
)

if not train_feature_scalers.empty:
    train_feature_scalers.to_csv(
        OUTPUT_DIR / "table_04_train_feature_scalers.csv",
        index=False,
    )


print("\nTrain volatility fit summary:")
display(train_volatility_fit)

print("\nTrain standardized-residual diagnostics:")
display(train_volatility_residual_diagnostics)


# ------------------------------------------------------------
# 6.2. Recursive one-step-ahead forecasts on validation
# ------------------------------------------------------------

validation_prediction_frames = []

for model_name, fitted in train_fitted_models.items():
    validation_prediction_frames.append(
        forecast_parametric_model(
            fitted_model=fitted,
            evaluation=valid_arma_output,
        )
    )

train_residuals = train_arma_residual_frame[
    "arma_resid"
]

validation_prediction_frames.extend([
    forecast_ewma(
        train_residuals=train_residuals,
        evaluation=valid_arma_output,
        lambda_=EWMA_LAMBDA,
    ),

    forecast_historical_variance(
        train_residuals=train_residuals,
        evaluation=valid_arma_output,
    ),

    forecast_rolling_variance(
        train_residuals=train_residuals,
        evaluation=valid_arma_output,
        window=22,
    ),
])

validation_predictions = pd.concat(
    validation_prediction_frames,
    ignore_index=True,
)

validation_metrics = evaluate_forecasts(
    predictions=validation_predictions,
    proxy_col="innovation_variance_proxy",
    sample_name="validation",
)


# ------------------------------------------------------------
# 6.3. Lock best model using validation QLIKE only
# ------------------------------------------------------------

eligible_validation = validation_metrics.loc[
    validation_metrics["eligible_for_ranking"]
].copy()

if eligible_validation.empty:
    raise RuntimeError(
        "No model is eligible for validation selection."
    )

LOCKED_VOLATILITY_MODEL = str(
    eligible_validation
    .sort_values(["QLIKE", "MSE"])
    .iloc[0]["model"]
)

validation_metrics[
    "validation_selected_locked_model"
] = (
    validation_metrics["model"]
    == LOCKED_VOLATILITY_MODEL
)

validation_predictions[
    "validation_selected_locked_model"
] = (
    validation_predictions["model"]
    == LOCKED_VOLATILITY_MODEL
)

validation_predictions.to_csv(
    OUTPUT_DIR / "table_05_validation_daily_forecasts.csv",
    index=False,
)

validation_metrics.to_csv(
    OUTPUT_DIR / "table_06_validation_accuracy.csv",
    index=False,
)

print(
    "Validation-selected locked model:",
    LOCKED_VOLATILITY_MODEL,
)

display(validation_metrics)


Training volatility fit | GARCH(1,1)-t
Training volatility fit | GJR-GARCH(1,1)-t
Training volatility fit | EGARCH(1,1)-t
Training volatility fit | Custom EGARCH no-X-t
Training volatility fit | EGARCH-X signed lag1-t
Training volatility fit | EGARCH-X abs lag1-t

Train volatility fit summary:


,model,converged,loglik,AIC,BIC,error
0,"EGARCH(1,1)-t",True,"2,654.800213","-5,299.600427","-5,269.545310",
1,Custom EGARCH no-X-t,True,"2,654.795416","-5,299.590833","-5,269.535716",
2,EGARCH-X signed lag1-t,True,"2,667.702146","-5,315.404293","-5,255.294059",
3,EGARCH-X abs lag1-t,True,"2,666.825974","-5,313.651948","-5,253.541714",
4,"GARCH(1,1)-t",True,"2,521.361729","-5,034.723459","-5,010.679365",
5,"GJR-GARCH(1,1)-t",True,"2,293.253388","-4,576.506776","-4,546.451659",



Train standardized-residual diagnostics:


,estimation_sample,model,n_std_resid,mean_std_resid,std_std_resid,max_abs_std_resid,n_abs_std_resid_gt3,LB_std_resid_p_lag10,LB_std_resid_p_lag20,LB_squared_std_resid_p_lag10,LB_squared_std_resid_p_lag20,ARCH_LM_p_lag10,diagnostic_pass,diagnostic_error
0,train,Custom EGARCH no-X-t,3014,-0.055730,1.221974,36.344547,35,0.089437,0.351754,0.999941,1.000000,0.999943,True,
1,train,"EGARCH(1,1)-t",3014,-0.055650,1.223328,36.450585,35,0.090523,0.355251,0.999944,1.000000,0.999946,True,
2,train,EGARCH-X abs lag1-t,3014,-0.064290,1.178111,34.641124,37,0.215309,0.412540,0.999999,1.000000,0.999999,True,
3,train,EGARCH-X signed lag1-t,3014,-0.065669,1.162179,31.554112,36,0.056785,0.156773,0.998681,1.000000,0.998720,True,
4,train,"GARCH(1,1)-t",3014,-0.065940,1.446338,38.354363,70,0.394610,0.730932,0.998068,1.000000,0.998063,True,
5,train,"GJR-GARCH(1,1)-t",3014,-0.039977,0.978743,18.481795,30,0.348975,0.718863,0.343011,0.927442,0.339879,True,


Validation-selected locked model: GJR-GARCH(1,1)-t


,sample,model,n_forecasts,eligible_for_ranking,QLIKE,MSE,RMSE,MAE,mean_forecast_variance,mean_actual_variance_proxy,QLIKE_rank,validation_selected_locked_model
0,validation,"GJR-GARCH(1,1)-t",501,True,-2.678337,0.030529,0.174725,0.053589,0.043310,0.041637,1.000000,True
1,validation,EGARCH-X abs lag1-t,501,True,-2.668385,0.029663,0.172228,0.061037,0.055230,0.041637,2.000000,False
2,validation,EGARCH-X signed lag1-t,501,True,-2.604624,0.028747,0.169549,0.053527,0.043440,0.041637,3.000000,False
3,validation,Custom EGARCH no-X-t,501,True,-2.569531,0.028835,0.169809,0.053460,0.043249,0.041637,4.000000,False
4,validation,"EGARCH(1,1)-t",501,True,-2.569411,0.028832,0.169800,0.053433,0.043205,0.041637,5.000000,False
5,validation,EWMA_0.94,501,True,-2.418535,0.029934,0.173013,0.054303,0.042313,0.041637,6.000000,False
6,validation,"GARCH(1,1)-t",501,True,-2.396691,0.028461,0.168703,0.045292,0.029278,0.041637,7.000000,False
7,validation,HistoricalVariance,501,True,-2.171211,0.031143,0.176475,0.061970,0.045956,0.041637,8.000000,False
8,validation,RollingVariance22,501,True,-2.077887,0.032209,0.179469,0.056715,0.042322,0.041637,9.000000,False


In [8]:

# ============================================================
# ROBUSTNESS: STANDARD GARCH DISTRIBUTION COMPARISON
# ============================================================

STANDARD_MODELS = [
    "GARCH(1,1)-t",
    "GJR-GARCH(1,1)-t",
    "EGARCH(1,1)-t",
]

DISTRIBUTIONS = [
    "normal",
    "t",
    "skewt",
]

distribution_fit_rows = []
distribution_forecast_frames = []
distribution_diagnostic_rows = []

for variance_specification in STANDARD_MODELS:
    base_name = variance_specification.replace(
        "-t",
        "",
    )

    for dist in DISTRIBUTIONS:
        candidate_name = f"{base_name}-{dist}"

        print("Distribution robustness |", candidate_name)

        try:
            fitted = fit_standard_volatility_model(
                model_name=variance_specification,
                train_residuals=train_arma_residual_frame[
                    "arma_resid"
                ],
                dist=dist,
            )

            # Keep recursion specification separate from display name.
            fitted["variance_specification"] = (
                variance_specification
            )

            fitted["model_name"] = (
                candidate_name
            )

            distribution_fit_rows.append({
                "model": candidate_name,
                "variance_specification": base_name,
                "distribution": dist,
                "converged": bool(
                    fitted["converged"]
                ),
                "loglik": float(
                    fitted["loglik"]
                ),
                "AIC": float(
                    fitted["AIC"]
                ),
                "BIC": float(
                    fitted["BIC"]
                ),
                "error": "",
            })

            try:
                distribution_diagnostic_rows.append(
                    volatility_residual_diagnostics(
                        fitted_model=fitted,
                        estimation_sample=(
                            "train_distribution_robustness"
                        ),
                    )
                )

            except Exception as diagnostic_exc:
                distribution_diagnostic_rows.append(
                    failed_volatility_diagnostic_row(
                        model_name=candidate_name,
                        estimation_sample=(
                            "train_distribution_robustness"
                        ),
                        error=str(diagnostic_exc),
                    )
                )

            if fitted["converged"]:
                distribution_forecast_frames.append(
                    forecast_standard_volatility(
                        fitted_model=fitted,
                        evaluation=valid_arma_output,
                    )
                )

        except Exception as exc:
            distribution_fit_rows.append({
                "model": candidate_name,
                "variance_specification": base_name,
                "distribution": dist,
                "converged": False,
                "loglik": np.nan,
                "AIC": np.nan,
                "BIC": np.nan,
                "error": str(exc)[:300],
            })

            distribution_diagnostic_rows.append(
                failed_volatility_diagnostic_row(
                    model_name=candidate_name,
                    estimation_sample=(
                        "train_distribution_robustness"
                    ),
                    error=str(exc),
                )
            )


distribution_fit_table = pd.DataFrame(
    distribution_fit_rows
)

distribution_residual_diagnostics = pd.DataFrame(
    distribution_diagnostic_rows
)

display(
    distribution_fit_table[
        [
            "model",
            "distribution",
            "converged",
            "AIC",
            "BIC",
            "error",
        ]
    ]
)

print("\nDistribution-robustness residual diagnostics:")
display(distribution_residual_diagnostics)


if not distribution_forecast_frames:
    raise RuntimeError(
        "No robustness model produced validation forecasts. "
        "Read the error column in distribution_fit_table."
    )


distribution_validation_predictions = pd.concat(
    distribution_forecast_frames,
    ignore_index=True,
)

distribution_validation_metrics = evaluate_forecasts(
    predictions=distribution_validation_predictions,
    proxy_col="innovation_variance_proxy",
    sample_name="validation_distribution_robustness",
)


distribution_fit_table.to_csv(
    OUTPUT_DIR / "robustness_standard_distribution_fit.csv",
    index=False,
)

distribution_residual_diagnostics.to_csv(
    OUTPUT_DIR
    / "robustness_standard_distribution_residual_diagnostics.csv",
    index=False,
)

distribution_validation_metrics.to_csv(
    OUTPUT_DIR
    / "robustness_standard_distribution_validation_accuracy.csv",
    index=False,
)


display(distribution_validation_metrics)


Distribution robustness | GARCH(1,1)-normal
Distribution robustness | GARCH(1,1)-t
Distribution robustness | GARCH(1,1)-skewt
Distribution robustness | GJR-GARCH(1,1)-normal
Distribution robustness | GJR-GARCH(1,1)-t
Distribution robustness | GJR-GARCH(1,1)-skewt
Distribution robustness | EGARCH(1,1)-normal
Distribution robustness | EGARCH(1,1)-t
Distribution robustness | EGARCH(1,1)-skewt


,model,distribution,converged,AIC,BIC,error
0,"GARCH(1,1)-normal",normal,True,"-3,226.371139","-3,208.338069",
1,"GARCH(1,1)-t",t,True,"-5,034.723459","-5,010.679365",
2,"GARCH(1,1)-skewt",skewt,True,"-4,640.313365","-4,610.258248",
3,"GJR-GARCH(1,1)-normal",normal,True,"-3,347.954704","-3,323.910611",
4,"GJR-GARCH(1,1)-t",t,True,"-4,576.506776","-4,546.451659",
5,"GJR-GARCH(1,1)-skewt",skewt,True,"-4,574.506776","-4,538.440636",
6,"EGARCH(1,1)-normal",normal,True,"-3,503.591592","-3,479.547499",
7,"EGARCH(1,1)-t",t,True,"-5,299.600427","-5,269.545310",
8,"EGARCH(1,1)-skewt",skewt,True,"-5,331.028077","-5,294.961937",



Distribution-robustness residual diagnostics:


,estimation_sample,model,n_std_resid,mean_std_resid,std_std_resid,max_abs_std_resid,n_abs_std_resid_gt3,LB_std_resid_p_lag10,LB_std_resid_p_lag20,LB_squared_std_resid_p_lag10,LB_squared_std_resid_p_lag20,ARCH_LM_p_lag10,diagnostic_pass,diagnostic_error
0,train_distribution_robustness,"GARCH(1,1)-normal",3014,-0.039291,1.058425,22.351938,35,0.144306,0.527784,0.005613,0.195493,0.005191,False,
1,train_distribution_robustness,"GARCH(1,1)-t",3014,-0.065940,1.446338,38.354363,70,0.394610,0.730932,0.998068,1.000000,0.998063,True,
2,train_distribution_robustness,"GARCH(1,1)-skewt",3014,-0.031334,1.003852,24.669171,27,0.126300,0.537927,0.036059,0.486409,0.035033,False,
3,train_distribution_robustness,"GJR-GARCH(1,1)-normal",3014,-0.042990,0.998489,22.307765,31,0.253299,0.512846,0.993073,0.999999,0.992950,True,
4,train_distribution_robustness,"GJR-GARCH(1,1)-t",3014,-0.039977,0.978743,18.481795,30,0.348975,0.718863,0.343011,0.927442,0.339879,True,
5,train_distribution_robustness,"GJR-GARCH(1,1)-skewt",3014,-0.039977,0.978743,18.481795,30,0.348975,0.718863,0.343011,0.927442,0.339879,True,
6,train_distribution_robustness,"EGARCH(1,1)-normal",3014,-0.052770,1.022000,18.961265,37,0.001066,0.007080,0.012094,0.265631,0.012259,False,
7,train_distribution_robustness,"EGARCH(1,1)-t",3014,-0.055650,1.223328,36.450585,35,0.090523,0.355251,0.999944,1.000000,0.999946,True,
8,train_distribution_robustness,"EGARCH(1,1)-skewt",3014,-0.051955,1.135948,33.544596,32,0.083606,0.333199,0.999925,1.000000,0.999928,True,


,sample,model,n_forecasts,eligible_for_ranking,QLIKE,MSE,RMSE,MAE,mean_forecast_variance,mean_actual_variance_proxy,QLIKE_rank
0,validation_distribution_robustness,"GJR-GARCH(1,1)-normal",501,True,-2.698429,0.031544,0.177607,0.057668,0.048797,0.041637,1.000000
1,validation_distribution_robustness,"GJR-GARCH(1,1)-t",501,True,-2.678337,0.030529,0.174725,0.053589,0.043310,0.041637,2.000000
2,validation_distribution_robustness,"GJR-GARCH(1,1)-skewt",501,True,-2.678337,0.030529,0.174725,0.053589,0.043310,0.041637,2.000000
3,validation_distribution_robustness,"GARCH(1,1)-normal",501,True,-2.607732,0.029127,0.170667,0.053236,0.042422,0.041637,4.000000
4,validation_distribution_robustness,"GARCH(1,1)-skewt",501,True,-2.597890,0.029161,0.170765,0.053586,0.042750,0.041637,5.000000
5,validation_distribution_robustness,"EGARCH(1,1)-skewt",501,True,-2.590144,0.029263,0.171064,0.057227,0.049030,0.041637,6.000000
6,validation_distribution_robustness,"EGARCH(1,1)-t",501,True,-2.569411,0.028832,0.169800,0.053433,0.043205,0.041637,7.000000
7,validation_distribution_robustness,"GARCH(1,1)-t",501,True,-2.396691,0.028461,0.168703,0.045292,0.029278,0.041637,8.000000
8,validation_distribution_robustness,"EGARCH(1,1)-normal",501,True,-2.387159,0.032341,0.179836,0.060343,0.048622,0.041637,9.000000


In [9]:

# ============================================================
# AUDIT CUSTOM EGARCH-X PARAMETERS
# ============================================================

custom_models = [
    "EGARCH-X signed lag1-t",
    "EGARCH-X abs lag1-t",
]

custom_parameter_audit_rows = []

for model_name in custom_models:
    if model_name not in train_fitted_models:
        custom_parameter_audit_rows.append({
            "model": model_name,
            "available": False,
            "beta_abs_below_0_999": False,
            "nu_above_2_05": False,
            "all_parameters_finite": False,
            "audit_pass": False,
            "error": "Model is unavailable because fitting failed.",
        })
        continue

    fitted = train_fitted_models[
        model_name
    ]

    params = fitted[
        "params"
    ]

    print("\nMODEL:", model_name)
    display(
        params.to_frame(
            "estimate"
        )
    )

    beta_abs_below_0_999 = bool(
        abs(
            float(
                params["beta"]
            )
        )
        < 0.999
    )

    nu_above_2_05 = bool(
        float(
            params["nu"]
        )
        > 2.05
    )

    all_parameters_finite = bool(
        np.isfinite(
            params.to_numpy(
                dtype=float
            )
        ).all()
    )

    audit_pass = (
        beta_abs_below_0_999
        and nu_above_2_05
        and all_parameters_finite
    )

    custom_parameter_audit_rows.append({
        "model": model_name,
        "available": True,
        "beta_abs_below_0_999": beta_abs_below_0_999,
        "nu_above_2_05": nu_above_2_05,
        "all_parameters_finite": all_parameters_finite,
        "audit_pass": audit_pass,
        "error": "",
    })


custom_parameter_audit = pd.DataFrame(
    custom_parameter_audit_rows
)

custom_parameter_audit.to_csv(
    OUTPUT_DIR
    / "table_04b_train_custom_egarchx_parameter_audit.csv",
    index=False,
)

display(custom_parameter_audit)

assert custom_parameter_audit[
    "audit_pass"
].all(), (
    "At least one custom EGARCH-X parameter audit failed. "
    "Inspect table_04b_train_custom_egarchx_parameter_audit.csv."
)

print("\nCustom EGARCH-X parameter audit: PASSED")



MODEL: EGARCH-X signed lag1-t


,estimate
omega,-0.042274
alpha,0.334902
gamma,0.011346
beta,0.977737
delta_vix_change_lag1,0.022564
delta_us10y_change_lag1,-0.029144
delta_dxy_return_lag1,0.046461
delta_oil_change_lag1,-0.056535
delta_vnindex_return_lag1,0.016668
nu,3.007098



MODEL: EGARCH-X abs lag1-t


,estimate
omega,-0.249859
alpha,0.418035
gamma,-0.010199
beta,0.953895
delta_vix_change_lag1,-0.028137
delta_us10y_change_lag1,0.052506
delta_dxy_return_lag1,0.064536
delta_oil_change_lag1,0.024957
delta_vnindex_return_lag1,0.046223
nu,3.026670


,model,available,beta_abs_below_0_999,nu_above_2_05,all_parameters_finite,audit_pass,error
0,EGARCH-X signed lag1-t,True,True,True,True,True,
1,EGARCH-X abs lag1-t,True,True,True,True,True,



Custom EGARCH-X parameter audit: PASSED


In [10]:
# ============================================================
# AUDIT VALIDATION FORECAST VARIANCE RANGE
# ============================================================

variance_range = (
    validation_predictions
    .groupby("model")["forecast_var"]
    .agg(
        min_forecast_var="min",
        median_forecast_var="median",
        mean_forecast_var="mean",
        max_forecast_var="max",
    )
    .sort_values("max_forecast_var", ascending=False)
)

display(variance_range)

assert np.isfinite(
    validation_predictions["forecast_var"]
).all(), "Non-finite forecast variance detected."

assert (
    validation_predictions["forecast_var"] > 0
).all(), "Non-positive forecast variance detected."

,min_forecast_var,median_forecast_var,mean_forecast_var,max_forecast_var
model,,,,
"GJR-GARCH(1,1)-t",0.011921,0.018961,0.043310,1.383191
EGARCH-X abs lag1-t,0.003608,0.028558,0.055230,0.887073
"GARCH(1,1)-t",0.002918,0.010848,0.029278,0.677497
EGARCH-X signed lag1-t,0.003293,0.022462,0.043440,0.597142
"EGARCH(1,1)-t",0.003333,0.022424,0.043205,0.547687
Custom EGARCH no-X-t,0.003330,0.022461,0.043249,0.547556
EWMA_0.94,0.003576,0.023703,0.042313,0.332822
RollingVariance22,0.002093,0.018602,0.042322,0.327642
HistoricalVariance,0.044460,0.045982,0.045956,0.047167


### **Refit locked model và report untouched test**

In [11]:
# ============================================================
# 7. REFIT ON TRAIN + VALIDATION -> UNTOUCHED TEST REPORT
# ============================================================

# ------------------------------------------------------------
# 7.1. Combine train and validation
# ------------------------------------------------------------

df_train_valid = (
    pd.concat(
        [df_train, df_valid],
        ignore_index=True,
    )
    .sort_values("Date")
    .reset_index(drop=True)
)

assert df_train_valid["Date"].max() < df_test["Date"].min(), (
    "Train + validation period overlaps with test period."
)

# ------------------------------------------------------------
# 7.2. Refit locked ARMA order on train + validation
# ------------------------------------------------------------

best_arma_fit_train_valid = fit_arma(
    y=df_train_valid["fx_return"].astype(float),
    p=LOCKED_ARMA_P,
    q=LOCKED_ARMA_Q,
)

train_valid_arma_residual_frame = build_arma_residual_frame(
    fitted_arma=best_arma_fit_train_valid,
    source_frame=df_train_valid,
)

test_arma_output = build_recursive_arma_innovations(
    fitted_arma=best_arma_fit_train_valid,
    evaluation_frame=df_test,
)

test_arma_output.to_csv(
    OUTPUT_DIR / "table_07_test_arma_innovations.csv",
    index=False,
)

print("Refitted ARMA order       :", LOCKED_ARMA_NAME)
print("Development innovations   :", len(train_valid_arma_residual_frame))
print("Untouched test observations:", len(test_arma_output))


# ------------------------------------------------------------
# 7.3. Define reporting models
# ------------------------------------------------------------

# Report all candidates on test, but do not reselect the locked model.
TEST_REPORT_MODELS = [
    LOCKED_VOLATILITY_MODEL,
    "GARCH(1,1)-t",
    "GJR-GARCH(1,1)-t",
    "EGARCH(1,1)-t",
    "Custom EGARCH no-X-t",
    "EGARCH-X signed lag1-t",
    "EGARCH-X abs lag1-t",
    f"EWMA_{EWMA_LAMBDA:.2f}",
    "HistoricalVariance",
    "RollingVariance22",
]

TEST_REPORT_MODELS = list(
    dict.fromkeys(TEST_REPORT_MODELS)
)

print("\nLocked validation-selected model:", LOCKED_VOLATILITY_MODEL)
print("Test reporting models           :", TEST_REPORT_MODELS)


# ------------------------------------------------------------
# 7.4. Refit parametric volatility models on train + validation
# ------------------------------------------------------------

development_fitted_models = {}
development_fit_rows = []
development_parameter_rows = []
development_scaler_rows = []
development_diagnostic_rows = []


for model_name in PARAMETRIC_MODELS:
    print("Development volatility refit |", model_name)

    try:
        fitted = fit_parametric_model(
            model_name=model_name,
            train_residual_frame=train_valid_arma_residual_frame,
        )

        development_fitted_models[model_name] = fitted

        development_fit_rows.append({
            "model": model_name,
            "converged": bool(fitted["converged"]),
            "loglik": float(fitted["loglik"]),
            "AIC": float(fitted["AIC"]),
            "BIC": float(fitted["BIC"]),
            "error": "",
        })

        for parameter, estimate in fitted["params"].items():
            development_parameter_rows.append({
                "model": model_name,
                "parameter": parameter,
                "estimate": float(estimate),
            })

        if fitted["kind"] == "custom_egarchx":
            for feature, mean, std in zip(
                SHOCK_COLS,
                fitted["scaler"]["mean"],
                fitted["scaler"]["std"],
            ):
                development_scaler_rows.append({
                    "model": model_name,
                    "feature": feature,
                    "mean": float(mean),
                    "std": float(std),
                })

        try:
            development_diagnostic_rows.append(
                volatility_residual_diagnostics(
                    fitted_model=fitted,
                    estimation_sample="train_plus_validation",
                )
            )

        except Exception as diagnostic_exc:
            development_diagnostic_rows.append(
                failed_volatility_diagnostic_row(
                    model_name=model_name,
                    estimation_sample="train_plus_validation",
                    error=str(diagnostic_exc),
                )
            )

    except Exception as exc:
        development_fit_rows.append({
            "model": model_name,
            "converged": False,
            "loglik": np.nan,
            "AIC": np.nan,
            "BIC": np.nan,
            "error": str(exc)[:300],
        })

        development_diagnostic_rows.append(
            failed_volatility_diagnostic_row(
                model_name=model_name,
                estimation_sample="train_plus_validation",
                error=str(exc),
            )
        )


development_volatility_fit = (
    pd.DataFrame(development_fit_rows)
    .sort_values(
        ["converged", "BIC"],
        ascending=[False, True],
        na_position="last",
    )
    .reset_index(drop=True)
)

development_volatility_parameters = pd.DataFrame(
    development_parameter_rows
)

development_feature_scalers = pd.DataFrame(
    development_scaler_rows
)

development_volatility_residual_diagnostics = (
    pd.DataFrame(
        development_diagnostic_rows
    )
    .sort_values(
        [
            "diagnostic_pass",
            "model",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)


development_volatility_fit.to_csv(
    OUTPUT_DIR / "table_08_development_volatility_fit.csv",
    index=False,
)

development_volatility_residual_diagnostics.to_csv(
    OUTPUT_DIR
    / "table_08b_development_volatility_residual_diagnostics.csv",
    index=False,
)

development_volatility_parameters.to_csv(
    OUTPUT_DIR / "table_09_development_volatility_parameters.csv",
    index=False,
)

if not development_feature_scalers.empty:
    development_feature_scalers.to_csv(
        OUTPUT_DIR / "table_10_development_feature_scalers.csv",
        index=False,
    )


display(development_volatility_fit)

print("\nDevelopment standardized-residual diagnostics:")
display(development_volatility_residual_diagnostics)


# ------------------------------------------------------------
# 7.5. Confirm locked model is available
# ------------------------------------------------------------

if LOCKED_VOLATILITY_MODEL in PARAMETRIC_MODELS:
    if LOCKED_VOLATILITY_MODEL not in development_fitted_models:
        raise RuntimeError(
            "Locked volatility model failed during train + validation refit."
        )

    if not development_fitted_models[
        LOCKED_VOLATILITY_MODEL
    ]["converged"]:
        raise RuntimeError(
            "Locked volatility model did not converge after refit."
        )


# ------------------------------------------------------------
# 7.6. Forecast untouched test period
# ------------------------------------------------------------

test_prediction_frames = []

development_residuals = (
    train_valid_arma_residual_frame["arma_resid"]
)


for model_name in TEST_REPORT_MODELS:

    # Parametric models
    if model_name in PARAMETRIC_MODELS:
        if model_name not in development_fitted_models:
            warnings.warn(
                f"Skip failed test-report model: {model_name}"
            )
            continue

        test_prediction_frames.append(
            forecast_parametric_model(
                fitted_model=development_fitted_models[model_name],
                evaluation=test_arma_output,
            )
        )

    # EWMA benchmark
    elif model_name == f"EWMA_{EWMA_LAMBDA:.2f}":
        test_prediction_frames.append(
            forecast_ewma(
                train_residuals=development_residuals,
                evaluation=test_arma_output,
                lambda_=EWMA_LAMBDA,
            )
        )

    # Expanding historical variance benchmark
    elif model_name == "HistoricalVariance":
        test_prediction_frames.append(
            forecast_historical_variance(
                train_residuals=development_residuals,
                evaluation=test_arma_output,
            )
        )

    # Rolling 22-day variance benchmark
    elif model_name == "RollingVariance22":
        test_prediction_frames.append(
            forecast_rolling_variance(
                train_residuals=development_residuals,
                evaluation=test_arma_output,
                window=22,
            )
        )

    else:
        raise ValueError(
            f"Unsupported test-report model: {model_name}"
        )


if not test_prediction_frames:
    raise RuntimeError(
        "No model produced untouched-test forecasts."
    )


test_predictions = pd.concat(
    test_prediction_frames,
    ignore_index=True,
)


# ------------------------------------------------------------
# 7.7. Evaluate untouched test forecasts
# ------------------------------------------------------------

test_metrics = evaluate_forecasts(
    predictions=test_predictions,
    proxy_col="innovation_variance_proxy",
    sample_name="test",
)

test_return_squared_robustness = evaluate_forecasts(
    predictions=test_predictions,
    proxy_col="return_squared_proxy",
    sample_name="test_return_squared_robustness",
)


# ------------------------------------------------------------
# 7.8. Mark locked model and reporting-only status
# ------------------------------------------------------------

for table in [
    test_predictions,
    test_metrics,
    test_return_squared_robustness,
]:
    table["validation_selected_locked_model"] = (
        table["model"] == LOCKED_VOLATILITY_MODEL
    )

    table["test_used_for_model_selection"] = False

    table["model_role"] = np.where(
        table["model"] == LOCKED_VOLATILITY_MODEL,
        "locked_validation_selected_model",
        "reporting_only_comparator",
    )


# ------------------------------------------------------------
# 7.9. Final test audit
# ------------------------------------------------------------

assert LOCKED_VOLATILITY_MODEL in set(
    test_metrics["model"]
), "Locked model is missing from untouched-test metrics."

assert (
    test_predictions["forecast_var"] > 0
).all(), "Non-positive test variance forecast detected."

assert np.isfinite(
    test_predictions["forecast_var"]
).all(), "Non-finite test variance forecast detected."

assert not test_metrics[
    "test_used_for_model_selection"
].any(), "Test set must not be used for model selection."


# ------------------------------------------------------------
# 7.10. Export final test results
# ------------------------------------------------------------

test_predictions.to_csv(
    OUTPUT_DIR / "table_11_test_daily_forecasts.csv",
    index=False,
)

test_metrics.to_csv(
    OUTPUT_DIR / "table_12_test_accuracy_locked_selection.csv",
    index=False,
)

test_return_squared_robustness.to_csv(
    OUTPUT_DIR / "table_13_test_return_squared_robustness.csv",
    index=False,
)


# ------------------------------------------------------------
# 7.11. Display reporting tables
# ------------------------------------------------------------

print("\nLocked model selected on validation:", LOCKED_VOLATILITY_MODEL)

print(
    "Test results are reporting-only. "
    "Do not reselect the model using test rankings."
)

print("\nUntouched-test accuracy using innovation-squared proxy:")
display(
    test_metrics.sort_values(
        ["QLIKE", "MSE"]
    )
)

print("\nRobustness ranking using raw return squared:")
display(
    test_return_squared_robustness.sort_values(
        ["QLIKE", "MSE"]
    )
)


Refitted ARMA order       : ARMA(4,5)
Development innovations   : 3515
Untouched test observations: 504

Locked validation-selected model: GJR-GARCH(1,1)-t
Test reporting models           : ['GJR-GARCH(1,1)-t', 'GARCH(1,1)-t', 'EGARCH(1,1)-t', 'Custom EGARCH no-X-t', 'EGARCH-X signed lag1-t', 'EGARCH-X abs lag1-t', 'EWMA_0.94', 'HistoricalVariance', 'RollingVariance22']
Development volatility refit | GARCH(1,1)-t
Development volatility refit | GJR-GARCH(1,1)-t
Development volatility refit | EGARCH(1,1)-t
Development volatility refit | Custom EGARCH no-X-t
Development volatility refit | EGARCH-X signed lag1-t
Development volatility refit | EGARCH-X abs lag1-t


,model,converged,loglik,AIC,BIC,error
0,Custom EGARCH no-X-t,True,"3,051.859202","-6,093.718404","-6,062.894430",
1,"EGARCH(1,1)-t",True,"3,051.858470","-6,093.716939","-6,062.892965",
2,EGARCH-X signed lag1-t,True,"3,063.598285","-6,107.196570","-6,045.548622",
3,EGARCH-X abs lag1-t,True,"3,059.594578","-6,099.189155","-6,037.541207",
4,"GARCH(1,1)-t",True,"2,648.068699","-5,288.137397","-5,263.478218",
5,"GJR-GARCH(1,1)-t",True,"2,635.683134","-5,261.366267","-5,230.542293",



Development standardized-residual diagnostics:


,estimation_sample,model,n_std_resid,mean_std_resid,std_std_resid,max_abs_std_resid,n_abs_std_resid_gt3,LB_std_resid_p_lag10,LB_std_resid_p_lag20,LB_squared_std_resid_p_lag10,LB_squared_std_resid_p_lag20,ARCH_LM_p_lag10,diagnostic_pass,diagnostic_error
0,train_plus_validation,Custom EGARCH no-X-t,3515,-0.028680,1.108247,35.620927,40,0.143197,0.364209,0.999999,1.000000,1.000000,True,
1,train_plus_validation,"EGARCH(1,1)-t",3515,-0.028696,1.108363,35.612337,40,0.143003,0.363689,0.999999,1.000000,0.999999,True,
2,train_plus_validation,EGARCH-X abs lag1-t,3515,-0.034736,1.077873,34.019170,40,0.231194,0.378093,1.000000,1.000000,1.000000,True,
3,train_plus_validation,EGARCH-X signed lag1-t,3515,-0.037252,1.053415,30.194111,41,0.077090,0.167954,0.999984,1.000000,0.999984,True,
4,train_plus_validation,"GJR-GARCH(1,1)-t",3515,-0.026036,0.981567,18.665732,38,0.290872,0.615552,0.090428,0.657237,0.087845,True,
5,train_plus_validation,"GARCH(1,1)-t",3515,-0.013834,0.998510,23.679066,37,0.065596,0.334072,0.000018,0.004618,0.000015,False,



Locked model selected on validation: GJR-GARCH(1,1)-t
Test results are reporting-only. Do not reselect the model using test rankings.

Untouched-test accuracy using innovation-squared proxy:


,sample,model,n_forecasts,eligible_for_ranking,QLIKE,MSE,RMSE,MAE,mean_forecast_variance,mean_actual_variance_proxy,QLIKE_rank,validation_selected_locked_model,test_used_for_model_selection,model_role
0,test,"EGARCH(1,1)-t",504,True,-2.851083,0.007986,0.089362,0.042545,0.040644,0.029276,1.000000,False,False,reporting_only_comparator
1,test,Custom EGARCH no-X-t,504,True,-2.851000,0.007987,0.089370,0.042557,0.040661,0.029276,2.000000,False,False,reporting_only_comparator
2,test,EGARCH-X abs lag1-t,504,True,-2.848452,0.008759,0.093590,0.043294,0.041015,0.029276,3.000000,False,False,reporting_only_comparator
3,test,EGARCH-X signed lag1-t,504,True,-2.848211,0.007952,0.089172,0.043454,0.042449,0.029276,4.000000,False,False,reporting_only_comparator
4,test,"GARCH(1,1)-t",504,True,-2.805900,0.007370,0.085851,0.037683,0.032599,0.029276,5.000000,False,False,reporting_only_comparator
5,test,"GJR-GARCH(1,1)-t",504,True,-2.775057,0.007868,0.088701,0.038095,0.032502,0.029276,6.000000,True,False,locked_validation_selected_model
6,test,EWMA_0.94,504,True,-2.759411,0.007477,0.086467,0.037472,0.030504,0.029276,7.000000,False,False,reporting_only_comparator
7,test,RollingVariance22,504,True,-2.696166,0.007819,0.088425,0.038091,0.030107,0.029276,8.000000,False,False,reporting_only_comparator
8,test,HistoricalVariance,504,True,-2.452284,0.007895,0.088854,0.049938,0.044945,0.029276,9.000000,False,False,reporting_only_comparator



Robustness ranking using raw return squared:


,sample,model,n_forecasts,eligible_for_ranking,QLIKE,MSE,RMSE,MAE,mean_forecast_variance,mean_actual_variance_proxy,QLIKE_rank,validation_selected_locked_model,test_used_for_model_selection,model_role
0,test_return_squared_robustness,EGARCH-X abs lag1-t,504,True,-2.897326,0.008477,0.092071,0.042730,0.041015,0.028035,1.000000,False,False,reporting_only_comparator
1,test_return_squared_robustness,"EGARCH(1,1)-t",504,True,-2.894699,0.007718,0.087850,0.042277,0.040644,0.028035,2.000000,False,False,reporting_only_comparator
2,test_return_squared_robustness,Custom EGARCH no-X-t,504,True,-2.894593,0.007719,0.087858,0.042289,0.040661,0.028035,3.000000,False,False,reporting_only_comparator
3,test_return_squared_robustness,EGARCH-X signed lag1-t,504,True,-2.894305,0.007651,0.087468,0.043234,0.042449,0.028035,4.000000,False,False,reporting_only_comparator
4,test_return_squared_robustness,"GARCH(1,1)-t",504,True,-2.847003,0.007085,0.084170,0.037626,0.032599,0.028035,5.000000,False,False,reporting_only_comparator
5,test_return_squared_robustness,"GJR-GARCH(1,1)-t",504,True,-2.820169,0.007537,0.086815,0.037621,0.032502,0.028035,6.000000,True,False,locked_validation_selected_model
6,test_return_squared_robustness,EWMA_0.94,504,True,-2.809762,0.007188,0.084784,0.037429,0.030504,0.028035,7.000000,False,False,reporting_only_comparator
7,test_return_squared_robustness,RollingVariance22,504,True,-2.763615,0.007536,0.086810,0.037952,0.030107,0.028035,8.000000,False,False,reporting_only_comparator
8,test_return_squared_robustness,HistoricalVariance,504,True,-2.479708,0.007613,0.087255,0.049836,0.044945,0.028035,9.000000,False,False,reporting_only_comparator


In [12]:
# ============================================================
# STAGE 4.1 — EXTRACT LOCKED GJR-GARCH DEVELOPMENT VOLATILITY
# ============================================================

locked_dev_model = development_fitted_models[
    LOCKED_VOLATILITY_MODEL
]

locked_dev_variance = np.asarray(
    locked_dev_model["in_sample_variance"],
    dtype=float,
)

assert len(locked_dev_variance) == len(train_valid_arma_residual_frame), (
    "Locked variance path is not aligned with development ARMA innovations."
)

assert np.isfinite(locked_dev_variance).all()
assert (locked_dev_variance > 0).all()

development_locked_volatility = (
    train_valid_arma_residual_frame[
        ["Date"]
    ]
    .copy()
    .reset_index(drop=True)
)

development_locked_volatility[
    "selected_cond_var"
] = locked_dev_variance

development_locked_volatility[
    "selected_cond_vol"
] = np.sqrt(
    development_locked_volatility[
        "selected_cond_var"
    ]
)

development_locked_volatility[
    "log_selected_cond_vol"
] = np.log(
    development_locked_volatility[
        "selected_cond_vol"
    ]
)

development_locked_volatility[
    "vol_source_model"
] = (
    f"{LOCKED_ARMA_NAME}-{LOCKED_VOLATILITY_MODEL}"
)

display(
    development_locked_volatility.head()
)

print(
    development_locked_volatility[
        "vol_source_model"
    ]
    .value_counts()
)

,Date,selected_cond_var,selected_cond_vol,log_selected_cond_vol,vol_source_model
0,2010-01-06,0.276212,0.525559,-0.643293,"ARMA(4,5)-GJR-GARCH(1,1)-t"
1,2010-01-07,0.170329,0.412710,-0.885011,"ARMA(4,5)-GJR-GARCH(1,1)-t"
2,2010-01-08,0.106843,0.326868,-1.118200,"ARMA(4,5)-GJR-GARCH(1,1)-t"
3,2010-01-11,0.069103,0.262874,-1.336082,"ARMA(4,5)-GJR-GARCH(1,1)-t"
4,2010-01-12,0.046092,0.214690,-1.538558,"ARMA(4,5)-GJR-GARCH(1,1)-t"


vol_source_model
ARMA(4,5)-GJR-GARCH(1,1)-t    3515
Name: count, dtype: int64


## Stage 4A — Supplementary EGARCH-X coefficient interpretation

The audited modeling notebook already refits both EGARCH-X candidate models on `train + validation`. Reuse those estimates.

These are supplementary comparator coefficients, not coefficients of the validation-locked forecasting model.

The existing modeling notebook exports point estimates only. Do not report p-values or significance stars unless standard errors are computed in a separate, explicitly documented inference block.

In [13]:
# ============================================================
# STAGE 4A — EGARCH-X DEVELOPMENT COEFFICIENT TABLE
# ============================================================

EGARCHX_MODELS = [
    "EGARCH-X signed lag1-t",
    "EGARCH-X abs lag1-t",
]

egarchx_development_coefficients = (
    development_volatility_parameters.loc[
        development_volatility_parameters[
            "model"
        ].isin(EGARCHX_MODELS)
    ]
    .copy()
    .sort_values(
        ["model", "parameter"]
    )
    .reset_index(drop=True)
)

assert not egarchx_development_coefficients.empty, (
    "Development EGARCH-X coefficients are missing. "
    "Run the train + validation refit block first."
)

egarchx_development_coefficients[
    "role"
] = "supplementary_comparator_not_locked_baseline"

display(
    egarchx_development_coefficients
)

,model,parameter,estimate,role
0,EGARCH-X abs lag1-t,alpha,0.467026,supplementary_comparator_not_locked_baseline
1,EGARCH-X abs lag1-t,beta,0.956588,supplementary_comparator_not_locked_baseline
2,EGARCH-X abs lag1-t,delta_dxy_return_lag1,0.044000,supplementary_comparator_not_locked_baseline
3,EGARCH-X abs lag1-t,delta_oil_change_lag1,0.017954,supplementary_comparator_not_locked_baseline
4,EGARCH-X abs lag1-t,delta_us10y_change_lag1,0.013987,supplementary_comparator_not_locked_baseline
5,EGARCH-X abs lag1-t,delta_vix_change_lag1,-0.019869,supplementary_comparator_not_locked_baseline
6,EGARCH-X abs lag1-t,delta_vnindex_return_lag1,0.049654,supplementary_comparator_not_locked_baseline
7,EGARCH-X abs lag1-t,gamma,-0.029358,supplementary_comparator_not_locked_baseline
8,EGARCH-X abs lag1-t,nu,2.688530,supplementary_comparator_not_locked_baseline
9,EGARCH-X abs lag1-t,omega,-0.152808,supplementary_comparator_not_locked_baseline


## Stage 4B — Dynamic HAC shock-volatility regressions

The dependent variable is `log_selected_cond_vol` from locked development-only GJR-GARCH volatility.

The EGARCH-X shock regressors are already lagged one trading day in the audited modeling pipeline:

```text
vix_change_lag1
us10y_change_lag1
dxy_return_lag1
oil_change_lag1
vnindex_return_lag1
```

Do not silently replace `oil_change_lag1` with `oil_return` or with a differently defined transformation.

In [14]:
# ============================================================
# STAGE 4B.1 — BUILD DEVELOPMENT-ONLY HAC FRAME
# ============================================================

HAC_LAGS = 5
HAC_Y_COL = "log_selected_cond_vol"
HAC_Y_LAG1_COL = "log_selected_cond_vol_lag1"

expected_shock_cols = [
    "vix_change_lag1",
    "us10y_change_lag1",
    "dxy_return_lag1",
    "oil_change_lag1",
    "vnindex_return_lag1",
]

assert SHOCK_COLS == expected_shock_cols, (
    "SHOCK_COLS changed relative to the audited modeling notebook."
)

hac_df = (
    train_valid_arma_residual_frame[
        ["Date", *SHOCK_COLS]
    ]
    .merge(
        development_locked_volatility[
            ["Date", HAC_Y_COL]
        ],
        on="Date",
        how="inner",
        validate="one_to_one",
    )
    .sort_values("Date")
    .reset_index(drop=True)
)

hac_df[
    HAC_Y_LAG1_COL
] = hac_df[
    HAC_Y_COL
].shift(1)

# Standardize lagged shock regressors on the development sample only.
signed_z_cols = []
abs_z_cols = []

for col in SHOCK_COLS:
    values = pd.to_numeric(
        hac_df[col],
        errors="coerce",
    )

    mean = float(values.mean())
    std = float(values.std(ddof=1))

    if not np.isfinite(std) or std <= 0:
        raise ValueError(
            f"Invalid development standard deviation for {col!r}."
        )

    z_col = f"z_{col}"
    abs_z_col = f"abs_z_{col}"

    hac_df[z_col] = (
        values - mean
    ) / std

    hac_df[abs_z_col] = (
        hac_df[z_col].abs()
    )

    signed_z_cols.append(z_col)
    abs_z_cols.append(abs_z_col)


# Optional development-only lagged crisis control.
hac_controls = []

if "crisis_dummy" in df_train_valid.columns:
    crisis_frame = (
        df_train_valid[
            ["Date", "crisis_dummy"]
        ]
        .copy()
        .sort_values("Date")
    )

    crisis_frame[
        "crisis_dummy_lag1"
    ] = pd.to_numeric(
        crisis_frame["crisis_dummy"],
        errors="coerce",
    ).shift(1)

    hac_df = (
        hac_df
        .merge(
            crisis_frame[
                ["Date", "crisis_dummy_lag1"]
            ],
            on="Date",
            how="left",
            validate="one_to_one",
        )
    )

    if (
        hac_df[
            "crisis_dummy_lag1"
        ]
        .nunique(dropna=True)
        > 1
    ):
        hac_controls.append(
            "crisis_dummy_lag1"
        )


hac_specs = {
    "HAC_dynamic_controls": [
        HAC_Y_LAG1_COL,
        *hac_controls,
    ],
    "HAC_dynamic_signed_lag1": [
        HAC_Y_LAG1_COL,
        *signed_z_cols,
        *hac_controls,
    ],
    "HAC_dynamic_abs_lag1": [
        HAC_Y_LAG1_COL,
        *abs_z_cols,
        *hac_controls,
    ],
}

all_hac_x_cols = sorted(
    {
        col
        for x_cols in hac_specs.values()
        for col in x_cols
    }
)

hac_common_df = (
    hac_df[
        [
            "Date",
            HAC_Y_COL,
            *all_hac_x_cols,
        ]
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan,
    )
    .dropna()
    .sort_values("Date")
    .reset_index(drop=True)
)

assert len(hac_common_df) > 50

print("HAC development rows :", len(hac_common_df))
print("HAC start date       :", hac_common_df["Date"].min())
print("HAC end date         :", hac_common_df["Date"].max())
print("HAC regressors       :", hac_specs)

HAC development rows : 3514
HAC start date       : 2010-01-07 00:00:00
HAC end date         : 2023-12-29 00:00:00
HAC regressors       : {'HAC_dynamic_controls': ['log_selected_cond_vol_lag1'], 'HAC_dynamic_signed_lag1': ['log_selected_cond_vol_lag1', 'z_vix_change_lag1', 'z_us10y_change_lag1', 'z_dxy_return_lag1', 'z_oil_change_lag1', 'z_vnindex_return_lag1'], 'HAC_dynamic_abs_lag1': ['log_selected_cond_vol_lag1', 'abs_z_vix_change_lag1', 'abs_z_us10y_change_lag1', 'abs_z_dxy_return_lag1', 'abs_z_oil_change_lag1', 'abs_z_vnindex_return_lag1']}


In [15]:
# ============================================================
# STAGE 4B.2 — HAC HELPERS
# ============================================================

def hac_ljung_box_pvalue(residuals, lag):
    residuals = (
        pd.Series(residuals)
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .dropna()
    )

    if len(residuals) <= lag + 5:
        return np.nan

    return float(
        acorr_ljungbox(
            residuals,
            lags=[lag],
            return_df=True,
        )["lb_pvalue"].iloc[0]
    )


def run_hac_regression(
    data,
    y_col,
    x_cols,
    model_name,
    hac_lags=5,
):
    x_cols = list(
        dict.fromkeys(x_cols)
    )

    d = (
        data[
            [
                y_col,
                *x_cols,
            ]
        ]
        .astype(float)
    )

    X = sm.add_constant(
        d[x_cols],
        has_constant="add",
    )

    y = d[y_col]

    if np.linalg.matrix_rank(
        X.to_numpy()
    ) != X.shape[1]:
        raise ValueError(
            f"{model_name}: rank-deficient design matrix."
        )

    res = (
        sm.OLS(
            y,
            X,
        )
        .fit(
            cov_type="HAC",
            cov_kwds={
                "maxlags": hac_lags,
                "use_correction": True,
            },
        )
    )

    coef_table = pd.DataFrame(
        {
            "model": model_name,
            "variable": res.params.index,
            "coef": res.params.values,
            "std_error_HAC": res.bse.values,
            "t_stat": res.tvalues.values,
            "p_value": res.pvalues.values,
        }
    )

    coef_table[
        "significance"
    ] = np.select(
        [
            coef_table["p_value"] < 0.01,
            coef_table["p_value"] < 0.05,
            coef_table["p_value"] < 0.10,
        ],
        [
            "***",
            "**",
            "*",
        ],
        default="",
    )

    summary_row = {
        "model": model_name,
        "volatility_source": (
            f"{LOCKED_ARMA_NAME}-{LOCKED_VOLATILITY_MODEL}"
        ),
        "nobs": int(res.nobs),
        "r2": float(res.rsquared),
        "adj_r2": float(res.rsquared_adj),
        "AIC": float(res.aic),
        "BIC": float(res.bic),
        "hac_lags": int(hac_lags),
        "durbin_watson": float(
            durbin_watson(res.resid)
        ),
        "LB_resid_p_lag10": (
            hac_ljung_box_pvalue(
                res.resid,
                10,
            )
        ),
        "LB_resid_p_lag20": (
            hac_ljung_box_pvalue(
                res.resid,
                20,
            )
        ),
        "x_cols": ", ".join(x_cols),
    }

    return res, coef_table, summary_row

In [16]:
# ============================================================
# STAGE 4B.3 — RUN AND EXPORT HAC MODELS
# ============================================================
from statsmodels.stats.stattools import durbin_watson
hac_models = {}
hac_coef_tables = []
hac_summary_rows = []

for model_name, x_cols in hac_specs.items():
    print("Estimating:", model_name)

    res, coef_table, summary_row = run_hac_regression(
        data=hac_common_df,
        y_col=HAC_Y_COL,
        x_cols=x_cols,
        model_name=model_name,
        hac_lags=HAC_LAGS,
    )

    hac_models[model_name] = res
    hac_coef_tables.append(coef_table)
    hac_summary_rows.append(summary_row)


hac_coef_table = pd.concat(
    hac_coef_tables,
    ignore_index=True,
)

hac_summary = (
    pd.DataFrame(hac_summary_rows)
    .sort_values(
        ["BIC", "AIC"]
    )
    .reset_index(drop=True)
)

display(hac_summary)
display(hac_coef_table)

Estimating: HAC_dynamic_controls
Estimating: HAC_dynamic_signed_lag1
Estimating: HAC_dynamic_abs_lag1


,model,volatility_source,nobs,r2,adj_r2,AIC,BIC,hac_lags,durbin_watson,LB_resid_p_lag10,LB_resid_p_lag20,x_cols
0,HAC_dynamic_controls,"ARMA(4,5)-GJR-GARCH(1,1)-t",3514,0.807854,0.807799,"-2,804.235031","-2,791.906011",5,1.932580,0.344897,0.008071,log_selected_cond_vol_lag1
1,HAC_dynamic_abs_lag1,"ARMA(4,5)-GJR-GARCH(1,1)-t",3514,0.808361,0.808033,"-2,803.515287","-2,760.363715",5,1.932128,0.354913,0.010417,"log_selected_cond_vol_lag1, abs_z_vix_change_l..."
2,HAC_dynamic_signed_lag1,"ARMA(4,5)-GJR-GARCH(1,1)-t",3514,0.808152,0.807824,"-2,799.701633","-2,756.550061",5,1.935122,0.340732,0.011779,"log_selected_cond_vol_lag1, z_vix_change_lag1,..."


,model,variable,coef,std_error_HAC,t_stat,p_value,significance
0,HAC_dynamic_controls,const,-0.195776,0.022211,-8.814452,0.000000,***
1,HAC_dynamic_controls,log_selected_cond_vol_lag1,0.897854,0.010985,81.735899,0.000000,***
2,HAC_dynamic_signed_lag1,const,-0.196558,0.022273,-8.824914,0.000000,***
3,HAC_dynamic_signed_lag1,log_selected_cond_vol_lag1,0.897446,0.011020,81.437840,0.000000,***
4,HAC_dynamic_signed_lag1,z_vix_change_lag1,-0.003090,0.002630,-1.174617,0.240148,
5,HAC_dynamic_signed_lag1,z_us10y_change_lag1,-0.001576,0.002968,-0.531163,0.595306,
6,HAC_dynamic_signed_lag1,z_dxy_return_lag1,-0.005153,0.002879,-1.789761,0.073492,*
7,HAC_dynamic_signed_lag1,z_oil_change_lag1,-0.000519,0.002197,-0.236293,0.813205,
8,HAC_dynamic_signed_lag1,z_vnindex_return_lag1,-0.000829,0.003256,-0.254606,0.799028,
9,HAC_dynamic_abs_lag1,const,-0.199551,0.024295,-8.213727,0.000000,***


## Stage 4C — Weekly recursive SVAR

Use contemporaneous daily shock transformations constructed upstream and aggregate them to weekly frequency. The weekly ordering remains:

```text
VIX → US10Y → DXY → OIL → VNINDEX → target
```

The target volatility series remains locked GJR-GARCH development volatility.

The audited forecasting notebook contains lagged daily shock regressors for EGARCH-X. Weekly SVAR requires the corresponding contemporaneous shock columns from the feature-engineering output. The cell fails explicitly when those columns are absent.

In [17]:
# ============================================================
# STAGE 4C.1 — BUILD DEVELOPMENT-ONLY WEEKLY DATA
#
# The audited split files contain lagged shocks only:
#     shock_lag1[t] = shock[t - 1]
#
# Weekly SVAR requires contemporaneous shocks:
#     shock[t] = shock_lag1.shift(-1)[t]
#
# The last development row is dropped because recovering its
# contemporaneous shock would require reading into untouched test.
# ============================================================

SVAR_SHOCKS = [
    "vix_change",
    "us10y_change",
    "dxy_return",
    "oil_change",
    "vnindex_return",
]

SVAR_TARGETS = [
    "fx_return",
    "log_selected_cond_vol",
]

CONTEMPORANEOUS_FROM_LAG1 = {
    "vix_change": "vix_change_lag1",
    "us10y_change": "us10y_change_lag1",
    "dxy_return": "dxy_return_lag1",
    "oil_change": "oil_change_lag1",
    "vnindex_return": "vnindex_return_lag1",
}

expected_lagged_cols = list(
    CONTEMPORANEOUS_FROM_LAG1.values()
)

missing_lagged_svar_cols = [
    col
    for col in [
        "Date",
        "fx_return",
        *expected_lagged_cols,
    ]
    if col not in df_train_valid.columns
]

if missing_lagged_svar_cols:
    raise KeyError(
        "Cannot reconstruct contemporaneous weekly-SVAR shocks. "
        "Missing audited development columns: "
        f"{missing_lagged_svar_cols}"
    )

svar_daily_base = (
    df_train_valid[
        [
            "Date",
            "fx_return",
            *expected_lagged_cols,
        ]
    ]
    .copy()
    .sort_values("Date")
    .reset_index(drop=True)
)

assert not svar_daily_base[
    "Date"
].duplicated().any()

assert svar_daily_base[
    "Date"
].is_monotonic_increasing

for contemporaneous_col, lagged_col in (
    CONTEMPORANEOUS_FROM_LAG1.items()
):
    svar_daily_base[
        contemporaneous_col
    ] = (
        pd.to_numeric(
            svar_daily_base[
                lagged_col
            ],
            errors="coerce",
        )
        .shift(-1)
    )

last_development_date = svar_daily_base[
    "Date"
].max()

svar_daily_base = (
    svar_daily_base
    .dropna(
        subset=[
            *SVAR_SHOCKS,
            "fx_return",
        ]
    )
    .copy()
)

assert (
    last_development_date
    not in set(
        svar_daily_base["Date"]
    )
)

daily_svar = (
    svar_daily_base[
        [
            "Date",
            *SVAR_SHOCKS,
            "fx_return",
        ]
    ]
    .merge(
        development_locked_volatility[
            [
                "Date",
                "log_selected_cond_vol",
            ]
        ],
        on="Date",
        how="inner",
        validate="one_to_one",
    )
    .replace(
        [np.inf, -np.inf],
        np.nan,
    )
    .dropna(
        subset=[
            *SVAR_SHOCKS,
            *SVAR_TARGETS,
        ]
    )
    .sort_values("Date")
    .set_index("Date")
)

assert len(daily_svar) > 100

assert daily_svar.index.max() < df_test[
    "Date"
].min()

weekly = pd.DataFrame(
    index=daily_svar
    .resample("W-FRI")
    .size()
    .index
)

weekly.index.name = "Date"

for col in [
    *SVAR_SHOCKS,
    "fx_return",
]:
    weekly[col] = (
        daily_svar[col]
        .resample("W-FRI")
        .sum(min_count=1)
    )

weekly[
    "log_selected_cond_vol"
] = (
    daily_svar[
        "log_selected_cond_vol"
    ]
    .resample("W-FRI")
    .mean()
)

weekly[
    "weekly_obs"
] = (
    daily_svar[
        "fx_return"
    ]
    .resample("W-FRI")
    .count()
)

MIN_WEEKLY_OBS = 2

weekly = (
    weekly.loc[
        weekly["weekly_obs"] >= MIN_WEEKLY_OBS
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan,
    )
    .dropna(
        subset=[
            *SVAR_SHOCKS,
            *SVAR_TARGETS,
        ]
    )
    .copy()
)

assert len(weekly) > 100

print("Weekly-SVAR shock source:")
print("  Explicit inverse alignment from audited lag-1 features")
print("  shock[t] = shock_lag1.shift(-1)[t]")
print("  Last development date excluded:", last_development_date)
print("Weekly rows       :", len(weekly))
print("Weekly start date :", weekly.index.min())
print("Weekly end date   :", weekly.index.max())

display(weekly.head())

Weekly-SVAR shock source:
  Explicit inverse alignment from audited lag-1 features
  shock[t] = shock_lag1.shift(-1)[t]
  Last development date excluded: 2023-12-29 00:00:00
Weekly rows       : 730
Weekly start date : 2010-01-08 00:00:00
Weekly end date   : 2023-12-29 00:00:00


,vix_change,us10y_change,dxy_return,oil_change,vnindex_return,fx_return,log_selected_cond_vol,weekly_obs
Date,,,,,,,,
2010-01-08,-1.220000,0.060000,-0.193438,1.000000,-2.208115,0.000000,-0.882168,3
2010-01-15,-0.220000,-0.130000,-0.193813,-4.780000,-3.016832,0.003293,-1.687567,5
2010-01-22,9.400000,-0.080000,1.233948,-3.710000,-5.663714,-0.003293,-2.129397,4
2010-01-29,-2.690000,0.010000,1.496161,-1.400000,0.910850,0.027307,-2.204661,5
2010-02-05,1.490000,-0.040000,1.225786,-1.700000,2.272918,0.000000,-2.213895,5


In [18]:
# ============================================================
# STAGE 4C.2 — STANDARDIZE WEEKLY VARIABLES
# ============================================================

weekly_model_cols = [
    *SVAR_SHOCKS,
    *SVAR_TARGETS,
]

weekly_means = weekly[
    weekly_model_cols
].mean()

weekly_stds = weekly[
    weekly_model_cols
].std(ddof=1)

invalid_weekly_stds = weekly_stds[
    (~np.isfinite(weekly_stds))
    | (weekly_stds <= 0)
]

if not invalid_weekly_stds.empty:
    raise ValueError(
        "Invalid weekly standard deviations: "
        f"{invalid_weekly_stds.index.tolist()}"
    )

weekly_z = (
    (
        weekly[
            weekly_model_cols
        ]
        - weekly_means
    )
    / weekly_stds
)

weekly_standardization = pd.DataFrame(
    {
        "variable": weekly_model_cols,
        "weekly_mean": weekly_means[
            weekly_model_cols
        ].values,
        "weekly_std": weekly_stds[
            weekly_model_cols
        ].values,
    }
)


display(weekly_standardization)

,variable,weekly_mean,weekly_std
0,vix_change,-0.009425,3.394441
1,us10y_change,0.000096,0.113681
2,dxy_return,0.036379,1.007609
3,oil_change,-0.013315,3.403721
4,vnindex_return,0.102930,2.719987
5,fx_return,0.037312,0.438146
6,log_selected_cond_vol,-1.914751,0.343827


In [19]:
# ============================================================
# STAGE 4C.3 — FIT WEEKLY RECURSIVE SVAR REPRESENTATION
#
# Rules:
# - Estimate reduced-form VAR models on weekly standardized data.
# - Select the lowest-BIC model only among stable candidates.
# - Never use an unstable VAR for Granger tests or IRFs.
# - For the volatility branch only, if the weekly log-volatility
#   level specification has no stable lag, try the transparent
#   fallback: first difference of weekly log-volatility.
# - Export lag diagnostics for every attempted specification.
# ============================================================

MAX_WEEKLY_VAR_LAG = 8
IRF_HORIZON = 12

weekly_svar_summary_rows = []
weekly_svar_granger_rows = []
weekly_svar_irf_rows = []
weekly_svar_lag_audit_tables = []
weekly_svar_models = {}


# ------------------------------------------------------------
# 1. Prepare explicit fallback target
# ------------------------------------------------------------

weekly_z_extended = weekly_z.copy()

weekly_z_extended[
    "delta_log_selected_cond_vol"
] = weekly_z_extended[
    "log_selected_cond_vol"
].diff()


# ------------------------------------------------------------
# 2. Candidate-fit helper
# ------------------------------------------------------------

def evaluate_var_candidates(
    data,
    maxlags=8,
    attempt_name="",
):
    """
    Fit lag orders 1..maxlags and retain a complete audit table.

    Returns
    -------
    selected_fit : VARResults or None
        Lowest-BIC stable candidate. None when no stable candidate exists.
    table : pd.DataFrame
        Complete candidate diagnostics, including failed fits.
    """
    rows = []
    stable_fits = {}

    for lag in range(1, maxlags + 1):
        try:
            fitted = VAR(data).fit(lag)

            roots = np.asarray(
                fitted.roots,
                dtype=complex,
            )

            min_abs_root = (
                float(np.min(np.abs(roots)))
                if roots.size > 0
                else np.nan
            )

            max_abs_companion_eigenvalue = (
                float(1.0 / min_abs_root)
                if np.isfinite(min_abs_root)
                and min_abs_root > 0
                else np.nan
            )

            stable = bool(
                fitted.is_stable()
            )

            rows.append({
                "attempt": attempt_name,
                "lag": lag,
                "nobs": int(fitted.nobs),
                "stable": stable,
                "min_abs_inverse_root": min_abs_root,
                "max_abs_companion_eigenvalue": (
                    max_abs_companion_eigenvalue
                ),
                "AIC": float(fitted.aic),
                "BIC": float(fitted.bic),
                "HQIC": float(fitted.hqic),
                "FPE": float(fitted.fpe),
                "error": "",
            })

            if stable:
                stable_fits[lag] = fitted

        except Exception as exc:
            rows.append({
                "attempt": attempt_name,
                "lag": lag,
                "nobs": np.nan,
                "stable": False,
                "min_abs_inverse_root": np.nan,
                "max_abs_companion_eigenvalue": np.nan,
                "AIC": np.nan,
                "BIC": np.nan,
                "HQIC": np.nan,
                "FPE": np.nan,
                "error": str(exc)[:500],
            })

    table = pd.DataFrame(rows)

    if not stable_fits:
        return None, table

    selected_lag = int(
        table.loc[
            table["lag"].isin(stable_fits)
            & table["stable"]
        ]
        .sort_values(
            ["BIC", "AIC", "lag"]
        )
        .iloc[0]["lag"]
    )

    return stable_fits[selected_lag], table


# ------------------------------------------------------------
# 3. Estimate target-specific recursive VAR representations
# ------------------------------------------------------------

SVAR_ATTEMPTS = {
    "fx_return": [
        {
            "target_col": "fx_return",
            "target_transformation": "weekly_level_return",
        },
    ],
    "log_selected_cond_vol": [
        {
            "target_col": "log_selected_cond_vol",
            "target_transformation": "weekly_log_volatility_level",
        },
        {
            "target_col": "delta_log_selected_cond_vol",
            "target_transformation": (
                "first_difference_of_weekly_log_volatility"
            ),
        },
    ],
}


for requested_target, attempts in SVAR_ATTEMPTS.items():
    fitted_var = None
    selected_target_col = None
    selected_transformation = None

    for attempt in attempts:
        target_col = attempt[
            "target_col"
        ]

        target_transformation = attempt[
            "target_transformation"
        ]

        ordered_cols = [
            *SVAR_SHOCKS,
            target_col,
        ]

        model_data = (
            weekly_z_extended[
                ordered_cols
            ]
            .replace(
                [np.inf, -np.inf],
                np.nan,
            )
            .dropna()
            .copy()
        )

        attempt_name = (
            f"{requested_target}__{target_transformation}"
        )

        print(
            "Weekly VAR stability audit |",
            attempt_name,
        )

        candidate_fit, lag_selection = (
            evaluate_var_candidates(
                data=model_data,
                maxlags=MAX_WEEKLY_VAR_LAG,
                attempt_name=attempt_name,
            )
        )

        lag_selection[
            "requested_target"
        ] = requested_target

        lag_selection[
            "estimated_target"
        ] = target_col

        lag_selection[
            "target_transformation"
        ] = target_transformation

        weekly_svar_lag_audit_tables.append(
            lag_selection
        )

        display(
            lag_selection[
                [
                    "attempt",
                    "lag",
                    "nobs",
                    "stable",
                    "max_abs_companion_eigenvalue",
                    "BIC",
                    "AIC",
                    "error",
                ]
            ]
        )

        if candidate_fit is not None:
            fitted_var = candidate_fit
            selected_target_col = target_col
            selected_transformation = (
                target_transformation
            )
            break

    model_name = (
        f"weekly_recursive_svar_shocks_to_{requested_target}"
    )

    if fitted_var is None:
        warnings.warn(
            f"Skip {model_name}: no stable weekly VAR candidate "
            "was found for any permitted transformation. "
            "Do not report Granger tests or IRFs for this branch."
        )

        weekly_svar_summary_rows.append({
            "model": model_name,
            "requested_target": requested_target,
            "estimated_target": "",
            "target_transformation": "",
            "n_obs_weekly": np.nan,
            "n_variables": len(SVAR_SHOCKS) + 1,
            "lag": np.nan,
            "ordering": "",
            "fit_status": "SKIPPED_no_stable_VAR",
            "var_is_stable": False,
            "volatility_source": (
                f"{LOCKED_ARMA_NAME}-{LOCKED_VOLATILITY_MODEL}"
                if requested_target
                == "log_selected_cond_vol"
                else ""
            ),
        })

        continue

    ordered_cols = [
        *SVAR_SHOCKS,
        selected_target_col,
    ]

    weekly_svar_models[
        model_name
    ] = fitted_var

    weekly_svar_summary_rows.append({
        "model": model_name,
        "requested_target": requested_target,
        "estimated_target": selected_target_col,
        "target_transformation": selected_transformation,
        "n_obs_weekly": int(fitted_var.nobs),
        "n_variables": len(ordered_cols),
        "lag": int(fitted_var.k_ar),
        "ordering": " -> ".join(ordered_cols),
        "fit_status": "OK",
        "var_is_stable": bool(
            fitted_var.is_stable()
        ),
        "volatility_source": (
            f"{LOCKED_ARMA_NAME}-{LOCKED_VOLATILITY_MODEL}"
            if requested_target
            == "log_selected_cond_vol"
            else ""
        ),
    })

    for shock in SVAR_SHOCKS:
        try:
            test = fitted_var.test_causality(
                caused=selected_target_col,
                causing=[shock],
                kind="f",
            )

            weekly_svar_granger_rows.append({
                "model": model_name,
                "requested_target": requested_target,
                "estimated_target": selected_target_col,
                "target_transformation": selected_transformation,
                "shock": shock,
                "lag": int(fitted_var.k_ar),
                "test_statistic": float(
                    test.test_statistic
                ),
                "p_value": float(
                    test.pvalue
                ),
                "significant_5pct": bool(
                    test.pvalue < 0.05
                ),
                "error": "",
            })

        except Exception as exc:
            weekly_svar_granger_rows.append({
                "model": model_name,
                "requested_target": requested_target,
                "estimated_target": selected_target_col,
                "target_transformation": selected_transformation,
                "shock": shock,
                "lag": int(fitted_var.k_ar),
                "test_statistic": np.nan,
                "p_value": np.nan,
                "significant_5pct": False,
                "error": str(exc)[:500],
            })

    # Orthogonalized responses under recursive Cholesky ordering.
    orth_irfs = fitted_var.irf(
        IRF_HORIZON
    ).orth_irfs

    target_index = ordered_cols.index(
        selected_target_col
    )

    for horizon in range(
        IRF_HORIZON + 1
    ):
        for shock_index, shock in enumerate(
            SVAR_SHOCKS
        ):
            weekly_svar_irf_rows.append({
                "model": model_name,
                "requested_target": requested_target,
                "estimated_target": selected_target_col,
                "target_transformation": selected_transformation,
                "shock": shock,
                "horizon_weeks": horizon,
                "orthogonalized_irf": float(
                    orth_irfs[
                        horizon,
                        target_index,
                        shock_index,
                    ]
                ),
            })


# ------------------------------------------------------------
# 4. Export complete audit and valid-model outputs
# ------------------------------------------------------------

weekly_svar_lag_audit = pd.concat(
    weekly_svar_lag_audit_tables,
    ignore_index=True,
)

weekly_svar_summary = pd.DataFrame(
    weekly_svar_summary_rows
)

weekly_svar_granger = pd.DataFrame(
    weekly_svar_granger_rows
)

weekly_svar_irf = pd.DataFrame(
    weekly_svar_irf_rows
)


print("Weekly recursive SVAR summary")
display(weekly_svar_summary)

print("Weekly recursive SVAR lag-selection audit")
display(weekly_svar_lag_audit)

print("Weekly recursive SVAR Granger-causality output")
display(weekly_svar_granger)

print("Weekly recursive SVAR IRF preview")
display(weekly_svar_irf.head(20))


Weekly VAR stability audit | fx_return__weekly_level_return


,attempt,lag,nobs,stable,max_abs_companion_eigenvalue,BIC,AIC,error
0,fx_return__weekly_level_return,1,729,True,0.139393,-0.230634,-0.495174,
1,fx_return__weekly_level_return,2,728,True,0.402893,0.001959,-0.489859,
2,fx_return__weekly_level_return,3,727,True,0.557488,0.229646,-0.489938,
3,fx_return__weekly_level_return,4,726,True,0.664908,0.499165,-0.448676,
4,fx_return__weekly_level_return,5,725,True,0.677144,0.783274,-0.393316,
5,fx_return__weekly_level_return,6,724,True,0.742930,0.985911,-0.419922,
6,fx_return__weekly_level_return,7,723,True,0.785681,1.210847,-0.424726,
7,fx_return__weekly_level_return,8,722,True,0.819716,1.420658,-0.445153,


Weekly VAR stability audit | log_selected_cond_vol__weekly_log_volatility_level


,attempt,lag,nobs,stable,max_abs_companion_eigenvalue,BIC,AIC,error
0,log_selected_cond_vol__weekly_log_volatility_l...,1,729,True,0.695663,-0.848317,-1.112858,
1,log_selected_cond_vol__weekly_log_volatility_l...,2,728,True,0.631628,-0.612732,-1.104550,
2,log_selected_cond_vol__weekly_log_volatility_l...,3,727,True,0.733251,-0.392403,-1.111987,
3,log_selected_cond_vol__weekly_log_volatility_l...,4,726,True,0.716059,-0.111019,-1.058860,
4,log_selected_cond_vol__weekly_log_volatility_l...,5,725,True,0.814979,0.165822,-1.010768,
5,log_selected_cond_vol__weekly_log_volatility_l...,6,724,True,0.843360,0.434271,-0.971563,
6,log_selected_cond_vol__weekly_log_volatility_l...,7,723,True,0.891448,0.633429,-1.002145,
7,log_selected_cond_vol__weekly_log_volatility_l...,8,722,True,0.895500,0.883442,-0.982369,


Weekly recursive SVAR summary


...

,attempt,lag,nobs,stable,min_abs_inverse_root,max_abs_companion_eigenvalue,AIC,BIC,HQIC,FPE,error,requested_target,estimated_target,target_transformation
0,fx_return__weekly_level_return,1,729,True,7.173963,0.139393,-0.495174,-0.230634,-0.393106,0.609467,,fx_return,fx_return,weekly_level_return
1,fx_return__weekly_level_return,2,728,True,2.482046,0.402893,-0.489859,0.001959,-0.300088,0.612727,,fx_return,fx_return,weekly_level_return
2,fx_return__weekly_level_return,3,727,True,1.793760,0.557488,-0.489938,0.229646,-0.212264,0.612708,,fx_return,fx_return,weekly_level_return
3,fx_return__weekly_level_return,4,726,True,1.503969,0.664908,-0.448676,0.499165,-0.082898,0.638577,,fx_return,fx_return,weekly_level_return
4,fx_return__weekly_level_return,5,725,True,1.476792,0.677144,-0.393316,0.783274,0.060766,0.675027,,fx_return,fx_return,weekly_level_return
5,fx_return__weekly_level_return,6,724,True,1.346022,0.742930,-0.419922,0.985911,0.122667,0.657449,,fx_return,fx_return,weekly_level_return
6,fx_return__weekly_level_return,7,723,True,1.272781,0.785681,-0.424726,1.210847,0.206573,0.654500,,fx_return,fx_return,weekly_level_return
7,fx_return__weekly_level_return,8,722,True,1.219935,0.819716,-0.445153,1.420658,0.275059,0.641530,,fx_return,fx_return,weekly_level_return
8,log_selected_cond_vol__weekly_log_volatility_l...,1,729,True,1.437478,0.695663,-1.112858,-0.848317,-1.010789,0.328620,,log_selected_cond_vol,log_selected_cond_vol,weekly_log_volatility_level
9,log_selected_cond_vol__weekly_log_volatility_l...,2,728,True,1.583211,0.631628,-1.104550,-0.612732,-0.914779,0.331368,,log_selected_cond_vol,log_selected_cond_vol,weekly_log_volatility_level


Weekly recursive SVAR Granger-causality output


,model,requested_target,estimated_target,target_transformation,shock,lag,test_statistic,p_value,significant_5pct,error
0,weekly_recursive_svar_shocks_to_fx_return,fx_return,fx_return,weekly_level_return,vix_change,1,1.333626,0.248225,False,
1,weekly_recursive_svar_shocks_to_fx_return,fx_return,fx_return,weekly_level_return,us10y_change,1,10.503247,0.001201,True,
2,weekly_recursive_svar_shocks_to_fx_return,fx_return,fx_return,weekly_level_return,dxy_return,1,6.792527,0.009185,True,
3,weekly_recursive_svar_shocks_to_fx_return,fx_return,fx_return,weekly_level_return,oil_change,1,0.492280,0.482950,False,
4,weekly_recursive_svar_shocks_to_fx_return,fx_return,fx_return,weekly_level_return,vnindex_return,1,4.121937,0.042392,True,
5,weekly_recursive_svar_shocks_to_log_selected_c...,log_selected_cond_vol,log_selected_cond_vol,weekly_log_volatility_level,vix_change,1,2.236175,0.134887,False,
6,weekly_recursive_svar_shocks_to_log_selected_c...,log_selected_cond_vol,log_selected_cond_vol,weekly_log_volatility_level,us10y_change,1,0.430071,0.511989,False,
7,weekly_recursive_svar_shocks_to_log_selected_c...,log_selected_cond_vol,log_selected_cond_vol,weekly_log_volatility_level,dxy_return,1,0.636411,0.425057,False,
8,weekly_recursive_svar_shocks_to_log_selected_c...,log_selected_cond_vol,log_selected_cond_vol,weekly_log_volatility_level,oil_change,1,2.288871,0.130378,False,
9,weekly_recursive_svar_shocks_to_log_selected_c...,log_selected_cond_vol,log_selected_cond_vol,weekly_log_volatility_level,vnindex_return,1,0.130720,0.717704,False,


Weekly recursive SVAR IRF preview


,model,requested_target,estimated_target,target_transformation,shock,horizon_weeks,orthogonalized_irf
0,weekly_recursive_svar_shocks_to_fx_return,fx_return,fx_return,weekly_level_return,vix_change,0,0.067675
1,weekly_recursive_svar_shocks_to_fx_return,fx_return,fx_return,weekly_level_return,us10y_change,0,0.037509
2,weekly_recursive_svar_shocks_to_fx_return,fx_return,fx_return,weekly_level_return,dxy_return,0,-0.004801
3,weekly_recursive_svar_shocks_to_fx_return,fx_return,fx_return,weekly_level_return,oil_change,0,-0.035865
4,weekly_recursive_svar_shocks_to_fx_return,fx_return,fx_return,weekly_level_return,vnindex_return,0,-0.118281
5,weekly_recursive_svar_shocks_to_fx_return,fx_return,fx_return,weekly_level_return,vix_change,1,0.051202
6,weekly_recursive_svar_shocks_to_fx_return,fx_return,fx_return,weekly_level_return,us10y_change,1,0.138510
7,weekly_recursive_svar_shocks_to_fx_return,fx_return,fx_return,weekly_level_return,dxy_return,1,0.105828
8,weekly_recursive_svar_shocks_to_fx_return,fx_return,fx_return,weekly_level_return,oil_change,1,-0.030887
9,weekly_recursive_svar_shocks_to_fx_return,fx_return,fx_return,weekly_level_return,vnindex_return,1,-0.066123


In [20]:
# ============================================================
# STAGE 4.4 — FINAL LEAKAGE AND SOURCE AUDIT
# ============================================================

assert (
    development_locked_volatility[
        "Date"
    ].max()
    < df_test[
        "Date"
    ].min()
), (
    "Stage 4 volatility target leaks into untouched test."
)

assert (
    weekly.index.max()
    < df_test[
        "Date"
    ].min()
), (
    "Stage 4 weekly SVAR sample leaks into untouched test."
)

assert (
    development_locked_volatility[
        "vol_source_model"
    ]
    == (
        f"{LOCKED_ARMA_NAME}-"
        f"{LOCKED_VOLATILITY_MODEL}"
    )
).all()

print("Stage 4 leakage audit: PASSED")
print(
    "Volatility source:",
    development_locked_volatility[
        "vol_source_model"
    ].iloc[0]
)
print(
    "Untouched test remains excluded from "
    "EGARCH-X interpretation, HAC, and weekly SVAR."
)

Stage 4 leakage audit: PASSED
Volatility source: ARMA(4,5)-GJR-GARCH(1,1)-t
Untouched test remains excluded from EGARCH-X interpretation, HAC, and weekly SVAR.


In [21]:

egarchx_development_coefficients.to_csv(
    OUTPUT_DIR
    / "table_16_egarchx_development_coefficients.csv",
    index=False,
)

hac_coef_table.to_csv(
    OUTPUT_DIR
    / "table_17_hac_shock_volatility_coefficients.csv",
    index=False,
)

hac_summary.to_csv(
    OUTPUT_DIR
    / "table_18_hac_shock_volatility_model_summary.csv",
    index=False,
)


weekly_standardization.to_csv(
    OUTPUT_DIR
    / "table_19_weekly_svar_standardization.csv",
    index=False,
)


lag_selection.to_csv(
    OUTPUT_DIR
    / f"table_20b_{model_name}_lag_selection.csv",
    index=False,
)


weekly_svar_lag_audit.to_csv(
    OUTPUT_DIR
    / "table_20b_weekly_recursive_svar_lag_selection_audit.csv",
    index=False,
)

weekly_svar_summary.to_csv(
    OUTPUT_DIR
    / "table_20_weekly_recursive_svar_summary.csv",
    index=False,
)

weekly_svar_granger.to_csv(
    OUTPUT_DIR
    / "table_21_weekly_recursive_svar_granger_causality.csv",
    index=False,
)

weekly_svar_irf.to_csv(
    OUTPUT_DIR
    / "table_22_weekly_recursive_svar_structural_irf.csv",
    index=False,
)
